# March 18 Einstein Crossing Times

This notebook:
- resolves the March 18 review DB and bundled light-curve paths using MALCA review modules
- loads the requested March 18 candidates from `output/runs/runs_march18_bundle_all/review/review.db`
- fits robust local models for each candidate using multiple center seeds
- compares flat, Gaussian, FRED, and full Paczynski models with BIC
- treats Gaussian as diagnostic only; only FRED can override Paczynski when it is favored by strong signed-log `\Delta`BIC evidence
- only reports `t_E` when the Paczynski fit passes quality cuts
- searches external microlensing catalogs via cone matches against OGLE EWS season tables, KMTNet event lists, MOA event tables, and Gaia Alerts
- uses sky separation as the primary external-match key, with fitted `t_0` and `t_E` agreement as secondary ranking information when the survey table exposes them
- plots each cleaned light curve with the fit and a residual panel

Notes:
- The primary center seed is the minimum-magnitude point in the cleaned light curve, i.e. the brightest point in the astronomical magnitude system.
- Additional seeds include the median time of the 5 brightest points and the pipeline `jump_best_t0` when available.
- Signed-log `\Delta`BIC is `sign(\Delta BIC) * log10(1 + |\Delta BIC|)`, so raw `|\Delta BIC| = 2, 6, 10` map to log values `0.477, 0.845, 1.041`.
- External microlensing tables prefer cached survey unions when available; OGLE/KMTNet `t_0` enrichment and Gaia Alerts matching fall back gracefully when network access is unavailable.
- `reported_tE_days` is set to `NaN` when the Paczynski fit is rejected.
- The request text had one missing comma; the candidate list below treats `609886176748` and `618475317371` as separate IDs.


In [1]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root (missing pyproject.toml).")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT


PosixPath('/home/calder/code/malca')

In [2]:
from __future__ import annotations

import sqlite3

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from astropy import units as u
from astropy.time import Time
from astropy.coordinates.solar_system import get_body_barycentric_posvel
from scipy.optimize import least_squares, lsq_linear

from malca.lightcurve_io import load_lightcurve_df
from malca.review.explore_data import infer_plot_dir_from_source
from malca.review.interactive_plot import resolve_lightcurve_path
from malca.review.store import get_candidate_payload
from malca.utils import clean_lc


MARCH18_CANDIDATE_IDS = [
    "120259784233",
    "489626721133",
    "481036788325",
    "68720699238",
    "77309955721",
    "326418117943",
    "541166175153",
    "188979054063",
    "25771219762",
    "575525833425",
    "472447489028",
    "103079263205",
    "34360800532",
    "171799355659",
    "627065322644",
    "609886176748",
    "618475317371",
    "566936418537",
    "77310050643",
]

DB_PATH = REPO_ROOT / "output" / "runs" / "runs_march18_bundle_all" / "review" / "review.db"
DB_PATH


PosixPath('/home/calder/code/malca/output/runs/runs_march18_bundle_all/review/review.db')

In [3]:
def _finite_float(value: object) -> float | None:
    try:
        number = float(value)
    except (TypeError, ValueError):
        return None
    return number if np.isfinite(number) else None


def _trend_baseline(t: np.ndarray, baseline: float, slope: float, t_ref: float) -> np.ndarray:
    return baseline + slope * (t - t_ref)


def _solve_u0_from_A0(A0: float) -> float:
    if A0 <= 1.0:
        return np.inf
    u0 = max(1.0 / A0, 1e-4)
    for _ in range(25):
        sqrt_term = np.sqrt(u0 * u0 + 4.0)
        A_curr = (u0 * u0 + 2.0) / (u0 * sqrt_term)
        eps = 1e-6
        up = u0 + eps
        um = max(u0 - eps, 1e-8)
        sp = np.sqrt(up * up + 4.0)
        sm = np.sqrt(um * um + 4.0)
        Ap = (up * up + 2.0) / (up * sp)
        Am = (um * um + 2.0) / (um * sm)
        dA_du = (Ap - Am) / (up - um)
        if not np.isfinite(dA_du) or dA_du == 0.0:
            break
        step = (A_curr - A0) / dA_du
        u0_next = max(u0 - step, 1e-8)
        if abs(u0_next - u0) < 1e-8:
            u0 = u0_next
            break
        u0 = u0_next
    return float(max(u0, 1e-8))


def _A0_from_u0(u0: float) -> float:
    u0 = max(abs(float(u0)), 1e-8)
    return float((u0 * u0 + 2.0) / (u0 * np.sqrt(u0 * u0 + 4.0)))


def paczynski_mag_trend(t: np.ndarray, A0: float, t0: float, tE: float, baseline: float, slope: float, t_ref: float) -> np.ndarray:
    trend = _trend_baseline(t, baseline, slope, t_ref)
    if A0 <= 1.0 or tE <= 0.0:
        return trend
    u0 = _solve_u0_from_A0(float(A0))
    u = np.sqrt(u0 * u0 + ((t - t0) / tE) ** 2)
    A = (u * u + 2.0) / (u * np.sqrt(u * u + 4.0))
    return trend - 2.5 * np.log10(A)


def gaussian_brightening_trend(t: np.ndarray, depth: float, t0: float, sigma: float, baseline: float, slope: float, t_ref: float) -> np.ndarray:
    trend = _trend_baseline(t, baseline, slope, t_ref)
    sigma = max(abs(float(sigma)), 1e-6)
    return trend - abs(depth) * np.exp(-0.5 * ((t - t0) / sigma) ** 2)


def fred_brightening_trend(t: np.ndarray, depth: float, t0: float, tau_rise: float, tau_decay: float, baseline: float, slope: float, t_ref: float) -> np.ndarray:
    trend = _trend_baseline(t, baseline, slope, t_ref)
    tau_rise = max(abs(float(tau_rise)), 1e-6)
    tau_decay = max(abs(float(tau_decay)), 1e-6)
    dt = t - t0
    rise_arg = np.clip(dt / tau_rise, -60.0, 60.0)
    decay_arg = np.clip(-dt / tau_decay, -60.0, 60.0)
    profile = np.where(dt < 0.0, np.exp(rise_arg), np.exp(decay_arg))
    return trend - abs(depth) * profile


def flat_trend(t: np.ndarray, baseline: float, slope: float, t_ref: float) -> np.ndarray:
    return _trend_baseline(t, baseline, slope, t_ref)


MODEL_PARAM_COUNTS = {
    'flat': 2,
    'gaussian': 5,
    'fred': 6,
    'paczynski': 5,
}

LOG10_DELTA_BIC_POSITIVE = float(np.log10(1.0 + 2.0))
LOG10_DELTA_BIC_STRONG = float(np.log10(1.0 + 6.0))
LOG10_DELTA_BIC_VERY_STRONG = float(np.log10(1.0 + 10.0))
NON_PACZYNSKI_SELECTION_LOG10_DELTA_BIC_THRESHOLD = LOG10_DELTA_BIC_STRONG

PARALLAX_MIN_TE_DAYS = 80.0
PARALLAX_MIN_FIT_POINTS = 80
PARALLAX_MIN_SPAN_DAYS = 240.0
PARALLAX_MAX_ABS_PIE = 1.5
PARALLAX_MAX_U0_ABS = 2.0
PARALLAX_MIN_U0_FACTOR = 1.0 / 3.0
PARALLAX_MAX_U0_FACTOR = 3.0
PARALLAX_MIN_TE_FACTOR = 0.35
PARALLAX_MAX_TE_FACTOR = 3.0
PARALLAX_BOUND_FRAC = 0.02
PARALLAX_MAX_REDUCED_CHI2 = 10.0
PARALLAX_REQUIRED_DELTA_CHI2 = 12.0
PARALLAX_REQUIRED_DELTA_BIC = 6.0
PARALLAX_ENABLE_MCMC = True
PARALLAX_MCMC_CHAINS = 6
PARALLAX_MCMC_BURN = 200
PARALLAX_MCMC_STEPS = 400
PARALLAX_MCMC_THIN = 2
PARALLAX_RANDOM_SEED = 20260322
PARALLAX_MIN_ACCEPTANCE_RATE = 0.002


def _signed_log10_delta_bic(value: object) -> float:
    numeric = _finite_float(value)
    if numeric is None:
        return np.nan
    return float(np.sign(numeric) * np.log10(1.0 + abs(numeric)))




def _mag_to_relative_flux(mag: np.ndarray, err_mag: np.ndarray, ref_mag: float | None = None) -> tuple[np.ndarray, np.ndarray, float]:
    mag = np.asarray(mag, dtype=float)
    err_mag = np.asarray(err_mag, dtype=float)
    if ref_mag is None or not np.isfinite(ref_mag):
        ref_mag = float(np.nanmedian(mag))
    flux = np.power(10.0, -0.4 * (mag - ref_mag))
    flux_err = (np.log(10.0) / 2.5) * flux * np.clip(err_mag, 1e-4, None)
    flux_err = np.clip(flux_err, 1e-8, None)
    return flux, flux_err, float(ref_mag)


def _relative_flux_to_mag(flux: np.ndarray, ref_mag: float) -> np.ndarray:
    flux = np.clip(np.asarray(flux, dtype=float), 1e-12, None)
    return float(ref_mag) - 2.5 * np.log10(flux)


def _pspl_magnification_from_tau_beta(tau: np.ndarray, beta: np.ndarray) -> np.ndarray:
    u = np.sqrt(np.maximum(np.asarray(tau, dtype=float) ** 2 + np.asarray(beta, dtype=float) ** 2, 1e-12))
    return (u * u + 2.0) / (u * np.sqrt(u * u + 4.0))


def _solve_source_blend_linear(magnification: np.ndarray, flux: np.ndarray, flux_err: np.ndarray) -> tuple[float, float, np.ndarray]:
    magnification = np.asarray(magnification, dtype=float)
    flux = np.asarray(flux, dtype=float)
    flux_err = np.asarray(flux_err, dtype=float)
    valid = np.isfinite(magnification) & np.isfinite(flux) & np.isfinite(flux_err) & (flux_err > 0.0)
    if int(np.sum(valid)) < 2:
        return np.nan, np.nan, np.full_like(flux, np.nan, dtype=float)

    A = magnification[valid]
    F = flux[valid]
    w = 1.0 / np.square(flux_err[valid])
    design = np.column_stack([A, np.ones_like(A)])
    design_w = design * np.sqrt(w[:, None])
    flux_w = F * np.sqrt(w)
    try:
        result = lsq_linear(design_w, flux_w, bounds=(0.0, np.inf), method='trf', lsmr_tol='auto')
        if not result.success or not np.all(np.isfinite(result.x)):
            raise RuntimeError(str(result.message))
        Fs, Fb = result.x
    except Exception:
        try:
            Fs, Fb = np.linalg.lstsq(design_w, flux_w, rcond=None)[0]
        except np.linalg.LinAlgError:
            return np.nan, np.nan, np.full_like(flux, np.nan, dtype=float)
        Fs = max(float(Fs), 0.0)
        Fb = max(float(Fb), 0.0)
    model = Fs * magnification + Fb
    return float(Fs), float(Fb), np.asarray(model, dtype=float)


def _sky_tangent_basis(ra_deg: float, dec_deg: float) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    ra = np.deg2rad(float(ra_deg))
    dec = np.deg2rad(float(dec_deg))
    east_hat = np.array([-np.sin(ra), np.cos(ra), 0.0], dtype=float)
    north_hat = np.array([-np.cos(ra) * np.sin(dec), -np.sin(ra) * np.sin(dec), np.cos(dec)], dtype=float)
    los_hat = np.array([np.cos(dec) * np.cos(ra), np.cos(dec) * np.sin(ra), np.sin(dec)], dtype=float)
    return east_hat, north_hat, los_hat


def _project_earth_orbit_geocentric(jd_full: np.ndarray, ra_deg: float, dec_deg: float, t0_ref_jd: float) -> dict[str, np.ndarray | float]:
    jd_full = np.asarray(jd_full, dtype=float)
    east_hat, north_hat, _ = _sky_tangent_basis(ra_deg, dec_deg)
    times_tdb = Time(jd_full, format='jd', scale='utc').tdb
    t0_ref_tdb = Time(float(t0_ref_jd), format='jd', scale='utc').tdb
    earth_pos, earth_vel = get_body_barycentric_posvel('earth', times_tdb)
    earth_pos_ref, earth_vel_ref = get_body_barycentric_posvel('earth', t0_ref_tdb)

    pos = earth_pos.xyz.to_value(u.au).T
    vel = earth_vel.xyz.to_value(u.au / u.day).T
    pos_ref = earth_pos_ref.xyz.to_value(u.au)
    vel_ref = earth_vel_ref.xyz.to_value(u.au / u.day)
    dt_days = times_tdb.jd - t0_ref_tdb.jd
    delta_vec = pos - pos_ref[None, :] - dt_days[:, None] * vel_ref[None, :]
    delta_n = delta_vec @ north_hat
    delta_e = delta_vec @ east_hat
    return {
        'jd_full': jd_full,
        't0_ref_jd': float(t0_ref_jd),
        'delta_n': np.asarray(delta_n, dtype=float),
        'delta_e': np.asarray(delta_e, dtype=float),
    }


def _microlens_param_bounds(
    jd_full: np.ndarray,
    *,
    u0_abs_guess: float | None = None,
    tE_guess_days: float | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    jd_full = np.asarray(jd_full, dtype=float)
    span = max(float(np.nanmax(jd_full) - np.nanmin(jd_full)), 30.0)
    tE_cap = min(max(25.0, 4.0 * span), 5000.0)

    u0_guess = _finite_float(u0_abs_guess)
    if u0_guess is None or u0_guess <= 0.0:
        u0_lo = 1e-3
        u0_hi = PARALLAX_MAX_U0_ABS
    else:
        u0_lo = max(1e-3, min(u0_guess * PARALLAX_MIN_U0_FACTOR, 0.95 * u0_guess))
        u0_hi = min(PARALLAX_MAX_U0_ABS, max(0.05, u0_guess * PARALLAX_MAX_U0_FACTOR))
        if u0_hi <= u0_lo:
            u0_hi = min(PARALLAX_MAX_U0_ABS, u0_lo * 1.5 + 0.05)

    tE_guess = _finite_float(tE_guess_days)
    if tE_guess is None or tE_guess <= 0.0:
        tE_lo = 5.0
        tE_hi = tE_cap
    else:
        tE_lo = max(5.0, tE_guess * PARALLAX_MIN_TE_FACTOR)
        tE_hi = min(tE_cap, max(30.0, tE_guess * PARALLAX_MAX_TE_FACTOR))
        if tE_hi <= tE_lo:
            tE_hi = min(tE_cap, tE_lo * 1.5 + 5.0)

    lower = np.array([np.log(u0_lo), np.nanmin(jd_full), np.log(tE_lo), -PARALLAX_MAX_ABS_PIE, -PARALLAX_MAX_ABS_PIE], dtype=float)
    upper = np.array([np.log(u0_hi), np.nanmax(jd_full), np.log(tE_hi), PARALLAX_MAX_ABS_PIE, PARALLAX_MAX_ABS_PIE], dtype=float)
    return lower, upper


def _profile_flux_microlensing_model(
    opt_params: np.ndarray,
    *,
    branch_sign: int,
    jd_full: np.ndarray,
    flux: np.ndarray,
    flux_err: np.ndarray,
    ref_mag: float,
    ephemeris: dict[str, np.ndarray | float] | None,
    with_parallax: bool,
) -> dict[str, object]:
    log_u0_abs = float(opt_params[0])
    t0_jd = float(opt_params[1])
    log_tE = float(opt_params[2])
    piE_n = float(opt_params[3]) if with_parallax and len(opt_params) > 3 else 0.0
    piE_e = float(opt_params[4]) if with_parallax and len(opt_params) > 4 else 0.0
    u0_abs = float(np.exp(log_u0_abs))
    u0 = float(branch_sign * u0_abs)
    tE_days = float(np.exp(log_tE))
    tau = (np.asarray(jd_full, dtype=float) - t0_jd) / tE_days
    beta = np.full_like(tau, u0, dtype=float)
    if with_parallax and ephemeris is not None:
        delta_n = np.asarray(ephemeris['delta_n'], dtype=float)
        delta_e = np.asarray(ephemeris['delta_e'], dtype=float)
        tau = tau + piE_n * delta_n + piE_e * delta_e
        beta = beta + piE_n * delta_e - piE_e * delta_n
    magnification = _pspl_magnification_from_tau_beta(tau, beta)
    Fs, Fb, model_flux = _solve_source_blend_linear(magnification, flux, flux_err)
    valid = np.isfinite(model_flux) & np.isfinite(flux) & np.isfinite(flux_err) & (flux_err > 0.0)
    residuals = np.full_like(flux, np.nan, dtype=float)
    if np.any(valid):
        residuals[valid] = (flux[valid] - model_flux[valid]) / flux_err[valid]
    chi2 = float(np.nansum(np.square(residuals[valid]))) if np.any(valid) else np.nan
    model_mag = _relative_flux_to_mag(model_flux, ref_mag)
    return {
        'u0': u0,
        'u0_abs': u0_abs,
        't0_jd': t0_jd,
        'tE_days': tE_days,
        'piE_N': piE_n,
        'piE_E': piE_e,
        'piE': float(np.hypot(piE_n, piE_e)),
        'Fs': float(Fs),
        'Fb': float(Fb),
        'magnification': magnification,
        'model_flux': model_flux,
        'model_mag': model_mag,
        'residuals': residuals,
        'chi2': chi2,
    }


def _fit_flux_microlensing_branch(
    *,
    jd_fit: np.ndarray,
    mag_fit: np.ndarray,
    err_fit: np.ndarray,
    ref_mag: float,
    branch_sign: int,
    t0_guess_jd: float,
    u0_abs_guess: float,
    tE_guess_days: float,
    with_parallax: bool,
    ephemeris: dict[str, np.ndarray | float] | None,
) -> dict[str, object]:
    jd_fit = np.asarray(jd_fit, dtype=float)
    mag_fit = np.asarray(mag_fit, dtype=float)
    err_fit = np.asarray(err_fit, dtype=float)
    flux_fit, flux_err_fit, ref_mag = _mag_to_relative_flux(mag_fit, err_fit, ref_mag=ref_mag)
    lower_full, upper_full = _microlens_param_bounds(jd_fit, u0_abs_guess=u0_abs_guess, tE_guess_days=tE_guess_days)
    if with_parallax:
        lower = lower_full.copy()
        upper = upper_full.copy()
    else:
        lower = lower_full[:3].copy()
        upper = upper_full[:3].copy()

    t0_guess_jd = float(np.clip(t0_guess_jd, lower_full[1], upper_full[1]))
    u0_abs_guess = float(np.clip(abs(u0_abs_guess), np.exp(lower_full[0]), np.exp(upper_full[0])))
    tE_guess_days = float(np.clip(abs(tE_guess_days), np.exp(lower_full[2]), np.exp(upper_full[2])))

    def residuals(opt_params: np.ndarray) -> np.ndarray:
        profile = _profile_flux_microlensing_model(
            opt_params,
            branch_sign=branch_sign,
            jd_full=jd_fit,
            flux=flux_fit,
            flux_err=flux_err_fit,
            ref_mag=ref_mag,
            ephemeris=ephemeris,
            with_parallax=with_parallax,
        )
        model_flux = np.asarray(profile['model_flux'], dtype=float)
        if (not np.all(np.isfinite(model_flux))) or (not np.isfinite(profile['Fs'])) or profile['Fs'] <= 0.0:
            return np.full_like(flux_fit, 1e6, dtype=float)
        if np.nanmin(model_flux) <= 0.0:
            return np.full_like(flux_fit, 1e6, dtype=float)
        return np.asarray(profile['residuals'], dtype=float)

    start_vectors: list[np.ndarray] = []
    if with_parallax:
        parallax_starts = [
            (0.0, 0.0),
            (0.10, 0.0),
            (-0.10, 0.0),
            (0.0, 0.10),
            (0.0, -0.10),
            (0.20, 0.20),
        ]
        for piE_n0, piE_e0 in parallax_starts:
            start_vectors.append(np.array([np.log(u0_abs_guess), t0_guess_jd, np.log(tE_guess_days), piE_n0, piE_e0], dtype=float))
        start_vectors.append(np.array([np.log(max(u0_abs_guess, 0.05)), t0_guess_jd, np.log(min(np.exp(upper_full[2]), 1.25 * tE_guess_days)), 0.0, 0.0], dtype=float))
    else:
        start_vectors.extend([
            np.array([np.log(u0_abs_guess), t0_guess_jd, np.log(tE_guess_days)], dtype=float),
            np.array([np.log(max(u0_abs_guess, 0.05)), t0_guess_jd, np.log(min(np.exp(upper_full[2]), 1.25 * tE_guess_days))], dtype=float),
            np.array([np.log(min(0.3, np.exp(upper_full[0]))), t0_guess_jd, np.log(max(np.exp(lower_full[2]), 0.75 * tE_guess_days))], dtype=float),
        ])

    best: dict[str, object] | None = None
    last_error = ''
    for x0 in start_vectors:
        try:
            result = least_squares(
                residuals,
                x0=np.clip(x0, lower + 1e-8, upper - 1e-8),
                bounds=(lower, upper),
                loss='linear',
                max_nfev=5000,
            )
        except Exception as exc:
            last_error = repr(exc)
            continue
        if not result.success or not np.all(np.isfinite(result.x)):
            last_error = str(result.message)
            continue
        profile = _profile_flux_microlensing_model(
            result.x,
            branch_sign=branch_sign,
            jd_full=jd_fit,
            flux=flux_fit,
            flux_err=flux_err_fit,
            ref_mag=ref_mag,
            ephemeris=ephemeris,
            with_parallax=with_parallax,
        )
        if (not np.isfinite(profile['chi2'])) or (not np.isfinite(profile['Fs'])) or profile['Fs'] <= 0.0:
            continue
        if best is None or float(profile['chi2']) < float(best['profile']['chi2']):
            best = {'result': result, 'profile': profile}

    if best is None:
        return {
            'success': False,
            'status': last_error or 'least_squares_failed',
            'branch_sign': int(branch_sign),
            'with_parallax': bool(with_parallax),
        }

    result = best['result']
    profile = best['profile']
    n_points = int(len(jd_fit))
    n_total_params = 7 if with_parallax else 5
    dof = max(n_points - n_total_params, 1)
    bic = float(profile['chi2'] + n_total_params * np.log(max(n_points, 2)))
    out = {
        'success': True,
        'status': 'ok',
        'branch_sign': int(branch_sign),
        'with_parallax': bool(with_parallax),
        'opt_params': np.asarray(result.x, dtype=float),
        'bounds': (np.asarray(lower, dtype=float), np.asarray(upper, dtype=float)),
        'least_squares_result': result,
        'chi2': float(profile['chi2']),
        'reduced_chi2': float(profile['chi2'] / dof),
        'bic': bic,
        'n_points': n_points,
        'ref_mag': float(ref_mag),
        **profile,
    }
    return out


def _build_branch_log_prob(
    *,
    branch_sign: int,
    jd_fit: np.ndarray,
    mag_fit: np.ndarray,
    err_fit: np.ndarray,
    ref_mag: float,
    ephemeris: dict[str, np.ndarray | float],
    lower: np.ndarray,
    upper: np.ndarray,
):
    flux_fit, flux_err_fit, ref_mag = _mag_to_relative_flux(mag_fit, err_fit, ref_mag=ref_mag)
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)

    def log_prob(opt_params: np.ndarray) -> float:
        opt_params = np.asarray(opt_params, dtype=float)
        if opt_params.shape != lower.shape:
            return -np.inf
        if np.any(opt_params <= lower) or np.any(opt_params >= upper):
            return -np.inf
        profile = _profile_flux_microlensing_model(
            opt_params,
            branch_sign=branch_sign,
            jd_full=jd_fit,
            flux=flux_fit,
            flux_err=flux_err_fit,
            ref_mag=ref_mag,
            ephemeris=ephemeris,
            with_parallax=True,
        )
        if (not np.isfinite(profile['chi2'])) or (not np.isfinite(profile['Fs'])) or profile['Fs'] <= 0.0:
            return -np.inf
        model_flux = np.asarray(profile['model_flux'], dtype=float)
        if np.nanmin(model_flux) <= 0.0:
            return -np.inf
        return float(-0.5 * profile['chi2'])

    return log_prob


def _run_metropolis_sampler(
    *,
    start: np.ndarray,
    lower: np.ndarray,
    upper: np.ndarray,
    log_prob,
    proposal_scale: np.ndarray,
    n_chains: int,
    n_burn: int,
    n_steps: int,
    thin: int,
    seed: int,
) -> dict[str, object]:
    rng = np.random.default_rng(int(seed))
    start = np.asarray(start, dtype=float)
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)
    proposal_scale = np.asarray(proposal_scale, dtype=float)
    proposal_scale = np.clip(proposal_scale, 1e-4, None)
    dim = int(len(start))
    kept_samples: list[np.ndarray] = []
    acceptance_rates: list[float] = []

    for _ in range(int(n_chains)):
        current = start.copy()
        current_logp = log_prob(current)
        for _attempt in range(50):
            if np.isfinite(current_logp):
                break
            trial = np.clip(start + rng.normal(scale=0.35 * proposal_scale, size=dim), lower + 1e-8, upper - 1e-8)
            trial_logp = log_prob(trial)
            if np.isfinite(trial_logp):
                current = trial
                current_logp = trial_logp
                break
        if not np.isfinite(current_logp):
            continue

        accepted = 0
        chain_samples: list[np.ndarray] = []
        total_steps = int(n_burn + n_steps)
        for istep in range(total_steps):
            proposal = np.clip(current + rng.normal(scale=proposal_scale, size=dim), lower + 1e-8, upper - 1e-8)
            proposal_logp = log_prob(proposal)
            if np.isfinite(proposal_logp):
                log_alpha = proposal_logp - current_logp
                if log_alpha >= 0.0 or np.log(rng.random()) < log_alpha:
                    current = proposal
                    current_logp = proposal_logp
                    accepted += 1
            if istep >= n_burn and ((istep - n_burn) % max(int(thin), 1) == 0):
                chain_samples.append(current.copy())

        if chain_samples:
            kept_samples.extend(chain_samples)
            acceptance_rates.append(accepted / max(total_steps, 1))

    if not kept_samples:
        return {
            'success': False,
            'n_samples': 0,
            'acceptance_rate': np.nan,
            'samples_opt': np.empty((0, dim), dtype=float),
        }

    return {
        'success': True,
        'n_samples': int(len(kept_samples)),
        'acceptance_rate': float(np.nanmean(acceptance_rates)) if acceptance_rates else np.nan,
        'samples_opt': np.asarray(kept_samples, dtype=float),
    }


def _summarize_branch_samples(samples_opt: np.ndarray, branch_sign: int) -> dict[str, float]:
    samples_opt = np.asarray(samples_opt, dtype=float)
    if samples_opt.size == 0:
        return {}
    u0_abs = np.exp(samples_opt[:, 0])
    t0_jd = samples_opt[:, 1]
    tE_days = np.exp(samples_opt[:, 2])
    piE_n = samples_opt[:, 3]
    piE_e = samples_opt[:, 4]
    u0 = branch_sign * u0_abs
    piE = np.hypot(piE_n, piE_e)

    def add_summary(out: dict[str, float], prefix: str, values: np.ndarray) -> None:
        q16, q50, q84 = np.nanpercentile(values, [16.0, 50.0, 84.0])
        out[f'{prefix}_p16'] = float(q16)
        out[f'{prefix}_p50'] = float(q50)
        out[f'{prefix}_p84'] = float(q84)

    out: dict[str, float] = {}
    add_summary(out, 'u0', u0)
    add_summary(out, 't0_jd', t0_jd)
    add_summary(out, 'tE_days', tE_days)
    add_summary(out, 'piE_N', piE_n)
    add_summary(out, 'piE_E', piE_e)
    add_summary(out, 'piE', piE)
    return out


def _parallax_seed_from_jacobian(branch_fit: dict[str, object], tE_days: float) -> np.ndarray:
    default_scale = np.array([0.03, max(0.6, 0.01 * max(float(tE_days), 1.0)), 0.03, 0.015, 0.015], dtype=float)
    result = branch_fit.get('least_squares_result')
    jac = getattr(result, 'jac', None) if result is not None else None
    if jac is None:
        return default_scale
    try:
        jac = np.asarray(jac, dtype=float)
        hess = jac.T @ jac
        cov = np.linalg.pinv(hess)
        scale = np.sqrt(np.clip(np.diag(cov), 1e-6, None))
        if scale.shape != default_scale.shape or not np.all(np.isfinite(scale)):
            return default_scale
        return np.clip(scale, 0.25 * default_scale, 2.0 * default_scale)
    except Exception:
        return default_scale


def _empty_parallax_result(status: str) -> dict[str, object]:
    return {
        'attempted': False,
        'fit_ok': False,
        'preferred': False,
        'status': status,
        't0_ref_jd': np.nan,
        'pspl': {},
        'branches': {'u0_pos': {}, 'u0_neg': {}},
        'best_branch': '',
        'delta_chi2': np.nan,
        'delta_bic': np.nan,
        'branch_delta_chi2': np.nan,
    }


def _param_near_bounds(value: float, lower: float, upper: float, frac: float = PARALLAX_BOUND_FRAC) -> bool:
    if not (np.isfinite(value) and np.isfinite(lower) and np.isfinite(upper) and upper > lower):
        return False
    width = upper - lower
    margin = max(float(frac) * width, 1e-6)
    return bool(value <= lower + margin or value >= upper - margin)


def _parallax_branch_quality(branch_fit: dict[str, object], pspl_fit: dict[str, object]) -> dict[str, object]:
    if not branch_fit.get('success'):
        return {'ok': False, 'warnings': ['fit_failed']}

    warnings: list[str] = []
    lower, upper = branch_fit.get('bounds', (None, None))
    opt_params = np.asarray(branch_fit.get('opt_params', []), dtype=float)
    lower_arr = np.asarray(lower, dtype=float) if lower is not None else np.asarray([])
    upper_arr = np.asarray(upper, dtype=float) if upper is not None else np.asarray([])
    param_names = ('log_u0_abs', 't0_jd', 'log_tE', 'piE_N', 'piE_E')
    if opt_params.shape == lower_arr.shape == upper_arr.shape:
        for idx, name in enumerate(param_names[: len(opt_params)]):
            if _param_near_bounds(float(opt_params[idx]), float(lower_arr[idx]), float(upper_arr[idx])):
                warnings.append(f'{name}_near_bound')

    pspl_tE = _finite_float(pspl_fit.get('tE_days'))
    par_tE = _finite_float(branch_fit.get('tE_days'))
    if pspl_tE is not None and par_tE is not None and pspl_tE > 0.0:
        ratio = par_tE / pspl_tE
        if ratio < 0.4 or ratio > 2.5:
            warnings.append('tE_shift_large')

    reduced_chi2 = _finite_float(branch_fit.get('reduced_chi2'))
    if reduced_chi2 is not None and reduced_chi2 > PARALLAX_MAX_REDUCED_CHI2:
        warnings.append('high_reduced_chi2')

    mcmc = branch_fit.get('mcmc', {}) or {}
    acceptance = _finite_float(mcmc.get('acceptance_rate'))
    if mcmc.get('success') and acceptance is not None and acceptance < PARALLAX_MIN_ACCEPTANCE_RATE:
        warnings.append('sampler_low_acceptance')
    return {'ok': len(warnings) == 0, 'warnings': warnings}


def _fit_parallax_diagnostics(
    context: dict[str, object],
    best_seed_result: dict[str, object],
    *,
    ra_deg: float | None,
    dec_deg: float | None,
) -> dict[str, object]:
    pac = best_seed_result['fits'].get('paczynski', {})
    if not pac.get('success'):
        return _empty_parallax_result('not_attempted:no_paczynski_fit')
    if ra_deg is None or dec_deg is None or not np.isfinite(ra_deg) or not np.isfinite(dec_deg):
        return _empty_parallax_result('not_attempted:missing_coordinates')

    jd_fit = np.asarray(best_seed_result['jd_fit'], dtype=float)
    mag_fit = np.asarray(best_seed_result['mag_fit'], dtype=float)
    err_fit = np.asarray(best_seed_result['err_fit'], dtype=float)
    if len(jd_fit) < PARALLAX_MIN_FIT_POINTS:
        return _empty_parallax_result('not_attempted:too_few_points')
    fit_span = float(np.nanmax(jd_fit) - np.nanmin(jd_fit)) if len(jd_fit) else 0.0
    if fit_span < PARALLAX_MIN_SPAN_DAYS:
        return _empty_parallax_result('not_attempted:fit_span_too_short')

    A0_guess = float(pac['params'][0])
    t0_guess_jd = float(pac['params'][1] + 2450000.0)
    tE_guess_days = float(abs(pac['params'][2]))
    if not np.isfinite(tE_guess_days) or tE_guess_days < PARALLAX_MIN_TE_DAYS:
        return _empty_parallax_result('not_attempted:tE_below_threshold')
    u0_abs_guess = float(np.clip(_solve_u0_from_A0(A0_guess), 1e-3, 3.0))
    ref_mag = float(np.nanmedian(mag_fit))
    jd_fit_full = jd_fit + 2450000.0

    pspl_fit = _fit_flux_microlensing_branch(
        jd_fit=jd_fit_full,
        mag_fit=mag_fit,
        err_fit=err_fit,
        ref_mag=ref_mag,
        branch_sign=+1,
        t0_guess_jd=t0_guess_jd,
        u0_abs_guess=u0_abs_guess,
        tE_guess_days=tE_guess_days,
        with_parallax=False,
        ephemeris=None,
    )
    if not pspl_fit.get('success'):
        out = _empty_parallax_result('fit_failed:pspl_flux_fit_failed')
        out['attempted'] = True
        return out

    ephemeris = _project_earth_orbit_geocentric(jd_fit_full, float(ra_deg), float(dec_deg), float(pspl_fit['t0_jd']))
    branches: dict[str, dict[str, object]] = {}
    for branch_sign, branch_name in ((+1, 'u0_pos'), (-1, 'u0_neg')):
        branch_fit = _fit_flux_microlensing_branch(
            jd_fit=jd_fit_full,
            mag_fit=mag_fit,
            err_fit=err_fit,
            ref_mag=ref_mag,
            branch_sign=branch_sign,
            t0_guess_jd=float(pspl_fit['t0_jd']),
            u0_abs_guess=float(abs(pspl_fit['u0'])),
            tE_guess_days=float(pspl_fit['tE_days']),
            with_parallax=True,
            ephemeris=ephemeris,
        )
        branch_fit['t0_ref_jd'] = float(pspl_fit['t0_jd'])
        if branch_fit.get('success') and PARALLAX_ENABLE_MCMC:
            lower, upper = branch_fit['bounds']
            log_prob = _build_branch_log_prob(
                branch_sign=branch_sign,
                jd_fit=jd_fit_full,
                mag_fit=mag_fit,
                err_fit=err_fit,
                ref_mag=ref_mag,
                ephemeris=ephemeris,
                lower=lower,
                upper=upper,
            )
            proposal_scale = _parallax_seed_from_jacobian(branch_fit, float(branch_fit['tE_days']))
            chain_seed = PARALLAX_RANDOM_SEED + int(context['candidate_id']) % 100000 + (0 if branch_sign > 0 else 500000)
            mcmc = _run_metropolis_sampler(
                start=np.asarray(branch_fit['opt_params'], dtype=float),
                lower=np.asarray(lower, dtype=float),
                upper=np.asarray(upper, dtype=float),
                log_prob=log_prob,
                proposal_scale=proposal_scale,
                n_chains=PARALLAX_MCMC_CHAINS,
                n_burn=PARALLAX_MCMC_BURN,
                n_steps=PARALLAX_MCMC_STEPS,
                thin=PARALLAX_MCMC_THIN,
                seed=chain_seed,
            )
            if mcmc.get('success'):
                mcmc['posterior_summary'] = _summarize_branch_samples(np.asarray(mcmc['samples_opt'], dtype=float), branch_sign)
            branch_fit['mcmc'] = mcmc
        else:
            branch_fit['mcmc'] = {'success': False, 'n_samples': 0, 'acceptance_rate': np.nan}
        branch_fit['quality'] = _parallax_branch_quality(branch_fit, pspl_fit)
        branches[branch_name] = branch_fit

    successful = {name: branch for name, branch in branches.items() if branch.get('success')}
    if not successful:
        return {
            'attempted': True,
            'fit_ok': False,
            'preferred': False,
            'status': 'fit_failed:parallax_branch_fit_failed',
            't0_ref_jd': float(pspl_fit['t0_jd']),
            'pspl': pspl_fit,
            'branches': branches,
            'best_branch': '',
            'delta_chi2': np.nan,
            'delta_bic': np.nan,
            'branch_delta_chi2': np.nan,
        }

    reliable = {name: branch for name, branch in successful.items() if branch.get('quality', {}).get('ok', False)}
    ranked = reliable or successful
    best_branch = min(ranked, key=lambda name: float(ranked[name]['chi2']))
    best_fit = ranked[best_branch]
    delta_chi2 = float(pspl_fit['chi2'] - best_fit['chi2']) if np.isfinite(pspl_fit.get('chi2', np.nan)) and np.isfinite(best_fit.get('chi2', np.nan)) else np.nan
    delta_bic = float(pspl_fit['bic'] - best_fit['bic']) if np.isfinite(pspl_fit.get('bic', np.nan)) and np.isfinite(best_fit.get('bic', np.nan)) else np.nan
    pos_chi2 = branches.get('u0_pos', {}).get('chi2', np.nan)
    neg_chi2 = branches.get('u0_neg', {}).get('chi2', np.nan)
    branch_delta_chi2 = float(abs(pos_chi2 - neg_chi2)) if np.isfinite(pos_chi2) and np.isfinite(neg_chi2) else np.nan
    fit_ok = bool(best_branch in reliable)
    warnings = list(best_fit.get('quality', {}).get('warnings', []))
    preferred = bool(fit_ok and np.isfinite(delta_chi2) and np.isfinite(delta_bic) and delta_chi2 >= PARALLAX_REQUIRED_DELTA_CHI2 and delta_bic >= PARALLAX_REQUIRED_DELTA_BIC)
    status = 'preferred' if preferred else ('fit_ok' if fit_ok else 'fit_unreliable')
    return {
        'attempted': True,
        'fit_ok': fit_ok,
        'preferred': preferred,
        'status': status,
        't0_ref_jd': float(pspl_fit['t0_jd']),
        'pspl': pspl_fit,
        'branches': branches,
        'best_branch': best_branch,
        'delta_chi2': delta_chi2,
        'delta_bic': delta_bic,
        'branch_delta_chi2': branch_delta_chi2,
        'warnings': warnings,
    }


def _evaluate_parallax_branch_mag(branch_fit: dict[str, object], jd_minus_2450000: np.ndarray, ra_deg: float, dec_deg: float) -> np.ndarray | None:
    if not branch_fit.get('success'):
        return None
    jd_full = np.asarray(jd_minus_2450000, dtype=float) + 2450000.0
    ephemeris = _project_earth_orbit_geocentric(jd_full, float(ra_deg), float(dec_deg), float(branch_fit['t0_ref_jd']))
    opt_params = np.asarray(branch_fit['opt_params'], dtype=float)
    profile = _profile_flux_microlensing_model(
        opt_params,
        branch_sign=int(branch_fit['branch_sign']),
        jd_full=jd_full,
        flux=np.ones_like(jd_full, dtype=float),
        flux_err=np.ones_like(jd_full, dtype=float),
        ref_mag=float(branch_fit['ref_mag']),
        ephemeris=ephemeris,
        with_parallax=True,
    )
    model_flux = float(branch_fit['Fs']) * np.asarray(profile['magnification'], dtype=float) + float(branch_fit['Fb'])
    return _relative_flux_to_mag(model_flux, float(branch_fit['ref_mag']))


def _flatten_parallax_summary(parallax_result: dict[str, object]) -> dict[str, object]:
    defaults: dict[str, object] = {
        'parallax_attempted': False,
        'parallax_fit_ok': False,
        'parallax_preferred': False,
        'parallax_status': parallax_result.get('status', ''),
        'parallax_warning': '',
        'parallax_t0_ref_jd': np.nan,
        'parallax_pspl_flux_chi2': np.nan,
        'parallax_pspl_flux_bic': np.nan,
        'parallax_pspl_flux_reduced_chi2': np.nan,
        'parallax_best_branch': '',
        'parallax_delta_chi2': np.nan,
        'parallax_delta_bic': np.nan,
        'parallax_branch_delta_chi2': np.nan,
        'parallax_best_t0_jd_minus_2450000': np.nan,
        'parallax_best_tE_days': np.nan,
        'parallax_best_u0': np.nan,
        'parallax_best_piE_N': np.nan,
        'parallax_best_piE_E': np.nan,
        'parallax_best_piE': np.nan,
        'parallax_best_fs': np.nan,
        'parallax_best_fb': np.nan,
        'parallax_best_chi2': np.nan,
        'parallax_best_reduced_chi2': np.nan,
        'parallax_best_bic': np.nan,
        'parallax_best_acceptance_rate': np.nan,
        'parallax_best_n_samples': 0,
    }
    for prefix in ('parallax_pos', 'parallax_neg'):
        defaults.update({
            f'{prefix}_t0_jd_minus_2450000': np.nan,
            f'{prefix}_tE_days': np.nan,
            f'{prefix}_u0': np.nan,
            f'{prefix}_piE_N': np.nan,
            f'{prefix}_piE_E': np.nan,
            f'{prefix}_piE': np.nan,
            f'{prefix}_chi2': np.nan,
            f'{prefix}_reduced_chi2': np.nan,
            f'{prefix}_bic': np.nan,
            f'{prefix}_acceptance_rate': np.nan,
            f'{prefix}_n_samples': 0,
        })

    out = defaults.copy()
    out['parallax_attempted'] = bool(parallax_result.get('attempted', False))
    out['parallax_fit_ok'] = bool(parallax_result.get('fit_ok', False))
    out['parallax_preferred'] = bool(parallax_result.get('preferred', False))
    out['parallax_status'] = str(parallax_result.get('status', '') or '')
    out['parallax_warning'] = ','.join(parallax_result.get('warnings', []) or [])
    out['parallax_t0_ref_jd'] = _finite_float(parallax_result.get('t0_ref_jd')) if _finite_float(parallax_result.get('t0_ref_jd')) is not None else np.nan
    out['parallax_best_branch'] = str(parallax_result.get('best_branch', '') or '')
    out['parallax_delta_chi2'] = _finite_float(parallax_result.get('delta_chi2')) if _finite_float(parallax_result.get('delta_chi2')) is not None else np.nan
    out['parallax_delta_bic'] = _finite_float(parallax_result.get('delta_bic')) if _finite_float(parallax_result.get('delta_bic')) is not None else np.nan
    out['parallax_branch_delta_chi2'] = _finite_float(parallax_result.get('branch_delta_chi2')) if _finite_float(parallax_result.get('branch_delta_chi2')) is not None else np.nan

    pspl = parallax_result.get('pspl', {}) or {}
    if pspl.get('success'):
        out['parallax_pspl_flux_chi2'] = float(pspl.get('chi2', np.nan))
        out['parallax_pspl_flux_bic'] = float(pspl.get('bic', np.nan))
        out['parallax_pspl_flux_reduced_chi2'] = float(pspl.get('reduced_chi2', np.nan))

    branch_map = {'u0_pos': 'parallax_pos', 'u0_neg': 'parallax_neg'}
    branches = parallax_result.get('branches', {}) or {}
    for branch_name, prefix in branch_map.items():
        branch = branches.get(branch_name, {}) or {}
        if not branch.get('success'):
            continue
        out[f'{prefix}_t0_jd_minus_2450000'] = float(branch.get('t0_jd', np.nan) - 2450000.0)
        out[f'{prefix}_tE_days'] = float(branch.get('tE_days', np.nan))
        out[f'{prefix}_u0'] = float(branch.get('u0', np.nan))
        out[f'{prefix}_piE_N'] = float(branch.get('piE_N', np.nan))
        out[f'{prefix}_piE_E'] = float(branch.get('piE_E', np.nan))
        out[f'{prefix}_piE'] = float(branch.get('piE', np.nan))
        out[f'{prefix}_chi2'] = float(branch.get('chi2', np.nan))
        out[f'{prefix}_reduced_chi2'] = float(branch.get('reduced_chi2', np.nan))
        out[f'{prefix}_bic'] = float(branch.get('bic', np.nan))
        out[f'{prefix}_acceptance_rate'] = float(branch.get('mcmc', {}).get('acceptance_rate', np.nan)) if branch.get('mcmc', {}).get('success') else np.nan
        out[f'{prefix}_n_samples'] = int(branch.get('mcmc', {}).get('n_samples', 0)) if branch.get('mcmc', {}).get('success') else 0

    best_name = out['parallax_best_branch']
    best = branches.get(best_name, {}) if best_name else {}
    if best and best.get('success'):
        out['parallax_best_t0_jd_minus_2450000'] = float(best.get('t0_jd', np.nan) - 2450000.0)
        out['parallax_best_tE_days'] = float(best.get('tE_days', np.nan))
        out['parallax_best_u0'] = float(best.get('u0', np.nan))
        out['parallax_best_piE_N'] = float(best.get('piE_N', np.nan))
        out['parallax_best_piE_E'] = float(best.get('piE_E', np.nan))
        out['parallax_best_piE'] = float(best.get('piE', np.nan))
        out['parallax_best_fs'] = float(best.get('Fs', np.nan))
        out['parallax_best_fb'] = float(best.get('Fb', np.nan))
        out['parallax_best_chi2'] = float(best.get('chi2', np.nan))
        out['parallax_best_reduced_chi2'] = float(best.get('reduced_chi2', np.nan))
        out['parallax_best_bic'] = float(best.get('bic', np.nan))
        if best.get('mcmc', {}).get('success'):
            out['parallax_best_acceptance_rate'] = float(best['mcmc'].get('acceptance_rate', np.nan))
            out['parallax_best_n_samples'] = int(best['mcmc'].get('n_samples', 0))
    return out

def _prepare_lightcurve_df(lc_path: Path, *, prefer_g_band: bool = True) -> tuple[pd.DataFrame, str]:
    df = load_lightcurve_df(lc_path)
    df = clean_lc(df)
    band_label = 'all'
    if prefer_g_band and 'v_g_band' in df.columns and (df['v_g_band'] == 0).any():
        df = df.loc[df['v_g_band'] == 0].copy()
        band_label = 'g'
    elif 'v_g_band' in df.columns and df['v_g_band'].nunique(dropna=True) == 1:
        band_label = 'g' if int(df['v_g_band'].iloc[0]) == 0 else 'V'
    return df.sort_values('JD').reset_index(drop=True), band_label


def _load_candidate_context(
    conn: sqlite3.Connection,
    candidate_id: str,
    *,
    plot_dir: Path | None,
    prefer_g_band: bool = True,
) -> dict[str, object]:
    row = pd.read_sql_query(
        """
        SELECT
            candidate_id,
            asas_sn_id,
            lc_path,
            source_path,
            jump_best_t0,
            jump_best_width_param,
            dip_best_t0,
            dip_best_width_param,
            baseline_mag,
            vetting_likely_known,
            catalog_source,
            vsx_class,
            asassn_var_type,
            gaia_var_class,
            ztf_var_type,
            simbad_otype,
            simbad_main_id,
            microlens_match,
            microlens_catalog,
            microlens_name,
            microlens_alt_name,
            microlens_te_days,
            microlens_sep_arcsec
        FROM candidates
        WHERE candidate_id = ?
        """,
        conn,
        params=[str(candidate_id)],
    )
    if row.empty:
        raise KeyError(f'Candidate not found in review DB: {candidate_id}')

    record = row.iloc[0].to_dict()
    payload = get_candidate_payload(conn, str(candidate_id))
    lc_path = resolve_lightcurve_path(payload, plot_dir)
    if lc_path is None:
        raise FileNotFoundError(f'Could not resolve light-curve path for {candidate_id}')
    df, band_label = _prepare_lightcurve_df(lc_path, prefer_g_band=prefer_g_band)
    if df.empty:
        raise ValueError(f'Resolved light curve is empty after cleaning: {candidate_id}')
    return {
        'candidate_id': str(record['candidate_id']),
        'asas_sn_id': str(record.get('asas_sn_id') or candidate_id),
        'row': record,
        'payload': payload,
        'lc_path': lc_path,
        'df': df,
        'band_label': band_label,
    }


def _bool_flag(value: object) -> bool:
    if value is None:
        return False
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    try:
        return bool(int(value))
    except (TypeError, ValueError):
        pass
    if isinstance(value, float) and np.isnan(value):
        return False
    return bool(value)


def _text_value(value: object) -> str:
    if value is None:
        return ''
    if isinstance(value, float) and np.isnan(value):
        return ''
    text = str(value).strip()
    return '' if text.lower() == 'nan' else text


def _format_known_microlens_status(result: dict[str, object]) -> str:
    summary = result['summary']
    lines = [f"Known-event status: {summary['candidate_id']}"]
    lines.append(f"  microlens_catalog_match: {'yes' if summary['microlens_match'] else 'no'}")
    if summary['microlens_match']:
        lines.append(f"  microlens_catalog: {summary['microlens_catalog'] or 'unknown'}")
        lines.append(f"  microlens_name: {summary['microlens_name'] or 'unknown'}")
        if summary['microlens_alt_name']:
            lines.append(f"  microlens_alt_name: {summary['microlens_alt_name']}")
        microlens_te_days = summary['microlens_te_days']
        microlens_sep_arcsec = summary['microlens_sep_arcsec']
        if microlens_te_days is not None and np.isfinite(microlens_te_days):
            lines.append(f"  published_microlens_tE_days: {microlens_te_days:.3f}")
        if microlens_sep_arcsec is not None and np.isfinite(microlens_sep_arcsec):
            lines.append(f"  microlens_sep_arcsec: {microlens_sep_arcsec:.3f}")
    else:
        lines.append('  microlens_catalog: none in review DB')

    lines.append(f"  vetting_likely_known: {'yes' if summary['vetting_likely_known'] else 'no'}")
    if summary['catalog_source']:
        lines.append(f"  catalog_source: {summary['catalog_source']}")

    class_bits = []
    if summary['vsx_class']:
        class_bits.append(f"VSX={summary['vsx_class']}")
    if summary['asassn_var_type']:
        class_bits.append(f"ASAS-SN={summary['asassn_var_type']}")
    if summary['gaia_var_class']:
        class_bits.append(f"Gaia={summary['gaia_var_class']}")
    if summary['ztf_var_type']:
        class_bits.append(f"ZTF={summary['ztf_var_type']}")
    if class_bits:
        lines.append('  catalog_classes: ' + '; '.join(class_bits))

    simbad_name = summary.get('nearest_simbad_object') or summary.get('simbad_main_id') or ''
    simbad_otype = summary.get('simbad_otype') or ''
    if simbad_name or simbad_otype:
        simbad_line = simbad_name or 'unknown'
        if simbad_otype:
            simbad_line += f" [{simbad_otype}]"
        lines.append(f"  simbad: {simbad_line}")
    return '\n'.join(lines)




def _format_parallax_status(result: dict[str, object]) -> str:
    parallax = result.get('parallax', {}) or {}
    summary = result.get('summary', {}) or {}
    status = str(summary.get('parallax_status', parallax.get('status', '')) or '')
    if not summary.get('parallax_attempted', False):
        return f"Parallax status: {status or 'not attempted'}"

    best_branch = summary.get('parallax_best_branch', '') or parallax.get('best_branch', '') or 'none'
    delta_chi2 = summary.get('parallax_delta_chi2')
    delta_bic = summary.get('parallax_delta_bic')
    delta_chi2_text = f"{float(delta_chi2):.2f}" if delta_chi2 is not None and np.isfinite(delta_chi2) else 'nan'
    delta_bic_text = f"{float(delta_bic):.2f}" if delta_bic is not None and np.isfinite(delta_bic) else 'nan'
    lines = [
        f"Parallax status: {status or 'attempted'} | preferred={bool(summary.get('parallax_preferred', False))} | best_branch={best_branch} | dchi2={delta_chi2_text} | dBIC={delta_bic_text}"
    ]
    if summary.get('parallax_warning'):
        lines.append(f"  warnings: {summary['parallax_warning']}")
    branch_map = (
        ('u0>0', 'parallax_pos', result.get('parallax', {}).get('branches', {}).get('u0_pos', {})),
        ('u0<0', 'parallax_neg', result.get('parallax', {}).get('branches', {}).get('u0_neg', {})),
    )
    for label, prefix, branch in branch_map:
        chi2 = summary.get(f'{prefix}_chi2')
        if chi2 is None or not np.isfinite(chi2):
            lines.append(f"  {label}: fit failed")
            continue
        lines.append(
            f"  {label}: chi2={float(chi2):.2f} | t0={float(summary.get(f'{prefix}_t0_jd_minus_2450000')):.2f} | "
            f"tE={float(summary.get(f'{prefix}_tE_days')):.2f} d | u0={float(summary.get(f'{prefix}_u0')):.4f} | "
            f"piE_N={float(summary.get(f'{prefix}_piE_N')):.3f} | piE_E={float(summary.get(f'{prefix}_piE_E')):.3f} | "
            f"|piE|={float(summary.get(f'{prefix}_piE')):.3f}"
        )
        posterior = branch.get('mcmc', {}).get('posterior_summary', {}) if branch else {}
        if posterior:
            lines.append(
                f"    posterior piE={posterior.get('piE_p50', np.nan):.3f} "
                f"[{posterior.get('piE_p16', np.nan):.3f}, {posterior.get('piE_p84', np.nan):.3f}] | "
                f"n={int(branch.get('mcmc', {}).get('n_samples', 0))}"
            )
        branch_warnings = branch.get('quality', {}).get('warnings', []) if branch else []
        if branch_warnings:
            lines.append(f"    warnings: {','.join(branch_warnings)}")
    return '\n'.join(lines)

def _pick_width_seed(row: dict[str, object]) -> float:
    candidates = []
    for key in ('jump_best_width_param', 'dip_best_width_param'):
        value = _finite_float(row.get(key))
        if value is not None and abs(value) > 0:
            candidates.append(abs(value))
    if candidates:
        return float(np.clip(np.nanmedian(candidates), 5.0, 300.0))
    return 40.0


def _candidate_seeds(df: pd.DataFrame, row: dict[str, object]) -> list[dict[str, object]]:
    if df.empty:
        raise ValueError('Cannot build a brightest5_median seed from an empty light curve')

    brightest = df.nsmallest(min(5, len(df)), 'mag')
    return [
        {
            'seed_method': 'brightest5_median',
            't0_guess': float(brightest['JD'].median()),
        }
    ]


def _estimate_return_half_window(jd: np.ndarray, mag: np.ndarray, err: np.ndarray, t0_guess: float, width_seed: float) -> float:
    base_half_window = float(np.clip(max(240.0, 8.0 * width_seed), 240.0, 1800.0))
    center_idx = int(np.nanargmin(np.abs(jd - t0_guess)))
    local_mask = np.abs(jd - t0_guess) <= max(120.0, 2.0 * width_seed)
    if int(local_mask.sum()) < 10:
        local_mask = np.ones_like(jd, dtype=bool)

    global_baseline = float(np.nanmedian(mag))
    local_min = float(np.nanmin(mag[local_mask]))
    depth_guess = max(global_baseline - local_min, 0.05)
    tol = max(0.15 * depth_guess, 3.0 * float(np.nanmedian(err)), 0.02)

    def find_return(direction: int) -> float | None:
        consec = 0
        idx_iter = range(center_idx, len(jd)) if direction > 0 else range(center_idx, -1, -1)
        for idx in idx_iter:
            near_baseline = abs(mag[idx] - global_baseline) <= tol
            consec = consec + 1 if near_baseline else 0
            if consec >= 3:
                anchor = idx - 2 if direction > 0 else idx + 2
                anchor = max(0, min(len(jd) - 1, anchor))
                return float(jd[anchor])
        return None

    left_return = find_return(-1)
    right_return = find_return(+1)

    distances = []
    if left_return is not None and left_return < t0_guess:
        distances.append(t0_guess - left_return)
    if right_return is not None and right_return > t0_guess:
        distances.append(right_return - t0_guess)

    if len(distances) == 2:
        half_window = max(distances) + max(60.0, 2.0 * width_seed)
    elif len(distances) == 1:
        half_window = max(base_half_window, distances[0] + max(120.0, 4.0 * width_seed))
    else:
        half_window = base_half_window
    return float(np.clip(half_window, 240.0, 2200.0))


def _outer_baseline_guess(jd_fit: np.ndarray, mag_fit: np.ndarray, center: float, half_window: float) -> tuple[float, float, float]:
    t_ref = float(np.nanmedian(jd_fit))
    outer_mask = np.abs(jd_fit - center) >= max(0.30 * half_window, 40.0)
    if int(outer_mask.sum()) >= 4:
        x = jd_fit[outer_mask] - t_ref
        y = mag_fit[outer_mask]
        slope, baseline = np.polyfit(x, y, deg=1)
        return float(baseline), float(slope), t_ref
    return float(np.nanmedian(mag_fit)), 0.0, t_ref


def _evaluate_model(model_name: str, params: np.ndarray, t: np.ndarray, t_ref: float) -> np.ndarray:
    if model_name == 'flat':
        baseline, slope = params
        return flat_trend(t, baseline, slope, t_ref)
    if model_name == 'gaussian':
        depth, t0, sigma, baseline, slope = params
        return gaussian_brightening_trend(t, depth, t0, sigma, baseline, slope, t_ref)
    if model_name == 'fred':
        depth, t0, tau_rise, tau_decay, baseline, slope = params
        return fred_brightening_trend(t, depth, t0, tau_rise, tau_decay, baseline, slope, t_ref)
    if model_name == 'paczynski':
        A0, t0, tE, baseline, slope = params
        return paczynski_mag_trend(t, A0, t0, tE, baseline, slope, t_ref)
    raise ValueError(f'Unknown model: {model_name}')


def _fit_model(
    model_name: str,
    jd_fit: np.ndarray,
    mag_fit: np.ndarray,
    err_fit: np.ndarray,
    *,
    center_guess: float,
    width_seed: float,
) -> dict[str, object]:
    half_window = 0.5 * float(jd_fit.max() - jd_fit.min())
    baseline_guess, slope_guess, t_ref = _outer_baseline_guess(jd_fit, mag_fit, center_guess, half_window)
    depth_guess = max(baseline_guess - float(np.nanmin(mag_fit)), 0.03)
    depth_limit = max(0.3, 3.0 * depth_guess)
    window_span = max(float(jd_fit.max() - jd_fit.min()), 1.0)
    slope_limit = max(0.005, 4.0 * depth_limit / window_span)

    if model_name == 'paczynski':
        best_result = None
        last_error = None

        lower_opt = np.array([np.log(1e-3), jd_fit.min(), np.log(0.5), baseline_guess - 1.5, -slope_limit], dtype=float)
        upper_opt = np.array([np.log(4.0), jd_fit.max(), np.log(max(window_span, 1.0)), baseline_guess + 1.5, slope_limit], dtype=float)

        brightest_idx = np.argsort(mag_fit)[:min(5, len(mag_fit))]
        t0_starts = [center_guess]
        if len(brightest_idx):
            t0_starts.append(float(np.median(jd_fit[brightest_idx])))
        t0_starts = list(dict.fromkeys(float(np.clip(val, jd_fit.min(), jd_fit.max())) for val in t0_starts))

        tE_starts = [
            max(5.0, 0.08 * window_span),
            max(5.0, 0.18 * window_span),
            max(5.0, 0.35 * window_span),
        ]
        tE_starts = list(dict.fromkeys(float(np.clip(val, 0.5, window_span)) for val in tE_starts))
        u0_starts = [0.02, 0.2, 1.0]

        def residuals_pacz(opt_params: np.ndarray) -> np.ndarray:
            log_u0, t0, log_tE, baseline, slope = opt_params
            A0 = _A0_from_u0(np.exp(log_u0))
            tE = float(np.exp(log_tE))
            model = paczynski_mag_trend(jd_fit, A0, t0, tE, baseline, slope, t_ref)
            return (mag_fit - model) / err_fit

        for t0_start in t0_starts:
            for u0_start in u0_starts:
                for tE_start in tE_starts:
                    x0 = np.array([np.log(u0_start), t0_start, np.log(tE_start), baseline_guess, slope_guess], dtype=float)
                    try:
                        result = least_squares(
                            residuals_pacz,
                            x0=np.clip(x0, lower_opt + 1e-8, upper_opt - 1e-8),
                            bounds=(lower_opt, upper_opt),
                            loss='soft_l1',
                            f_scale=1.5,
                            max_nfev=2500,
                        )
                    except Exception as exc:
                        last_error = repr(exc)
                        continue

                    if not result.success or not np.all(np.isfinite(result.x)):
                        last_error = result.message
                        continue

                    trial_resid = residuals_pacz(result.x)
                    trial_chi2 = float(np.nansum(trial_resid ** 2))
                    if best_result is None or trial_chi2 < best_result['chi2']:
                        best_result = {'result': result, 'chi2': trial_chi2}

        if best_result is None:
            return {'model_name': model_name, 'success': False, 'error': last_error or 'Paczynski multistart failed'}

        result = best_result['result']
        log_u0, t0, log_tE, baseline, slope = result.x.astype(float)
        params = np.array([_A0_from_u0(np.exp(log_u0)), t0, np.exp(log_tE), baseline, slope], dtype=float)
        lower = np.array([_A0_from_u0(np.exp(upper_opt[0])), lower_opt[1], np.exp(lower_opt[2]), lower_opt[3], lower_opt[4]], dtype=float)
        upper = np.array([_A0_from_u0(np.exp(lower_opt[0])), upper_opt[1], np.exp(upper_opt[2]), upper_opt[3], upper_opt[4]], dtype=float)
    else:
        if model_name == 'flat':
            p0 = np.array([baseline_guess, slope_guess], dtype=float)
            lower = np.array([baseline_guess - 1.5, -slope_limit], dtype=float)
            upper = np.array([baseline_guess + 1.5, slope_limit], dtype=float)
        elif model_name == 'gaussian':
            p0 = np.array([depth_guess, center_guess, max(width_seed, 5.0), baseline_guess, slope_guess], dtype=float)
            lower = np.array([0.01, jd_fit.min(), 0.2, baseline_guess - 1.5, -slope_limit], dtype=float)
            upper = np.array([depth_limit, jd_fit.max(), window_span, baseline_guess + 1.5, slope_limit], dtype=float)
        elif model_name == 'fred':
            tau0 = max(width_seed, 5.0)
            p0 = np.array([depth_guess, center_guess, tau0, tau0, baseline_guess, slope_guess], dtype=float)
            lower = np.array([0.01, jd_fit.min(), 0.2, 0.2, baseline_guess - 1.5, -slope_limit], dtype=float)
            upper = np.array([depth_limit, jd_fit.max(), window_span, window_span, baseline_guess + 1.5, slope_limit], dtype=float)
        else:
            raise ValueError(model_name)

        def residuals(params: np.ndarray) -> np.ndarray:
            model = _evaluate_model(model_name, params, jd_fit, t_ref)
            return (mag_fit - model) / err_fit

        try:
            result = least_squares(
                residuals,
                x0=np.clip(p0, lower + 1e-8, upper - 1e-8),
                bounds=(lower, upper),
                loss='soft_l1',
                f_scale=1.5,
                max_nfev=8000,
            )
        except Exception as exc:
            return {'model_name': model_name, 'success': False, 'error': repr(exc)}

        if not result.success or not np.all(np.isfinite(result.x)):
            return {
                'model_name': model_name,
                'success': False,
                'error': result.message,
            }

        params = result.x.astype(float)

    model = _evaluate_model(model_name, params, jd_fit, t_ref)
    resid = (mag_fit - model) / err_fit
    chi2 = float(np.nansum(resid ** 2))
    n = int(len(jd_fit))
    k = MODEL_PARAM_COUNTS[model_name]
    dof = max(n - k, 1)
    reduced_chi2 = chi2 / dof
    bic = chi2 + k * np.log(max(n, 2))

    return {
        'model_name': model_name,
        'success': True,
        'params': params,
        'bounds': (lower.astype(float), upper.astype(float)),
        'model': model,
        'residuals': resid,
        'chi2': chi2,
        'reduced_chi2': reduced_chi2,
        'bic': bic,
        'n_points': n,
        't_ref': t_ref,
        'least_squares_cost': float(result.cost),
        'message': result.message,
    }


def _fit_model_suite(df: pd.DataFrame, *, center_guess: float, width_seed: float) -> dict[str, object]:
    jd = df['JD'].to_numpy(dtype=float)
    mag = df['mag'].to_numpy(dtype=float)
    err = np.clip(df['error'].to_numpy(dtype=float), 0.01, None)

    half_window = _estimate_return_half_window(jd, mag, err, center_guess, width_seed)
    fit_mask = np.abs(jd - center_guess) <= half_window
    if int(fit_mask.sum()) < 30:
        half_window = max(half_window, 360.0)
        fit_mask = np.abs(jd - center_guess) <= half_window
    if int(fit_mask.sum()) < 20:
        fit_mask = np.ones_like(jd, dtype=bool)
        half_window = 0.5 * float(jd.max() - jd.min())

    def run_all(current_mask: np.ndarray, current_center: float, current_width_seed: float) -> dict[str, object]:
        jd_fit = jd[current_mask]
        mag_fit = mag[current_mask]
        err_fit = err[current_mask]
        fits = {}
        for model_name in ('flat', 'gaussian', 'fred', 'paczynski'):
            fits[model_name] = _fit_model(
                model_name,
                jd_fit,
                mag_fit,
                err_fit,
                center_guess=current_center,
                width_seed=current_width_seed,
            )
        return {
            'fit_mask': current_mask.copy(),
            'half_window': float(half_window),
            'jd_fit': jd_fit,
            'mag_fit': mag_fit,
            'err_fit': err_fit,
            'fits': fits,
        }

    suite = run_all(fit_mask, center_guess, width_seed)
    pac = suite['fits']['paczynski']
    if pac.get('success'):
        pac_params = pac['params']
        pac_t0 = float(pac_params[1])
        pac_tE = float(abs(pac_params[2]))
        refined_half = max(half_window, _estimate_return_half_window(jd, mag, err, pac_t0, max(width_seed, 8.0 * pac_tE)), 10.0 * pac_tE)
        refined_half = float(np.clip(refined_half, 240.0, 2200.0))
        refined_mask = np.abs(jd - pac_t0) <= refined_half
        if int(refined_mask.sum()) >= int(np.sum(fit_mask)) + 10:
            half_window = refined_half
            suite = run_all(refined_mask, pac_t0, max(width_seed, pac_tE))

    fits = suite['fits']
    comparison_model_names = ('flat', 'gaussian', 'fred', 'paczynski')
    curve_alt_model_names = ('gaussian', 'fred')
    successful = {name: fit for name, fit in fits.items() if name in comparison_model_names and fit.get('success')}
    best_model = min(successful, key=lambda name: successful[name]['bic']) if successful else None

    pac = successful.get('paczynski')
    gaussian_fit = fits.get('gaussian', {})
    curve_alts = {name: fit for name, fit in successful.items() if name in curve_alt_model_names}
    best_alt_model = min(curve_alts, key=lambda name: curve_alts[name]['bic']) if curve_alts else None
    best_alt_bic = float(curve_alts[best_alt_model]['bic']) if best_alt_model is not None else np.nan
    delta_bic_vs_gaussian = np.nan
    delta_bic_vs_fred = np.nan
    if pac is not None and gaussian_fit.get('success'):
        delta_bic_vs_gaussian = float(gaussian_fit['bic'] - pac['bic'])
    if pac is not None and 'fred' in successful:
        delta_bic_vs_fred = float(successful['fred']['bic'] - pac['bic'])
    delta_bic_vs_flat = np.nan
    delta_bic_vs_best_alt = np.nan
    if pac is not None and 'flat' in successful:
        delta_bic_vs_flat = float(successful['flat']['bic'] - pac['bic'])
    if pac is not None and np.isfinite(best_alt_bic):
        delta_bic_vs_best_alt = float(best_alt_bic - pac['bic'])

    suite['best_model'] = best_model
    suite['best_alt_model'] = best_alt_model
    suite['delta_bic_vs_flat'] = delta_bic_vs_flat
    suite['delta_bic_vs_gaussian'] = delta_bic_vs_gaussian
    suite['delta_bic_vs_fred'] = delta_bic_vs_fred
    suite['delta_bic_vs_best_alt'] = delta_bic_vs_best_alt
    suite['best_alt_bic'] = best_alt_bic
    return suite


def _pacz_quality_metrics(seed_result: dict[str, object]) -> dict[str, object]:
    pac = seed_result['fits'].get('paczynski', {})
    if not pac.get('success'):
        return {
            'fit_ok': False,
            'warnings': ['paczynski_fit_failed'],
            'shoulder_left': 0,
            'shoulder_right': 0,
            'n_strong_points': 0,
        }

    jd_fit = seed_result['jd_fit']
    mag_fit = seed_result['mag_fit']
    pac_model = pac['model']
    A0, t0, tE, baseline, slope = pac['params']
    lower, upper = pac['bounds']
    t_ref = pac['t_ref']
    baseline_model = flat_trend(jd_fit, baseline, slope, t_ref)
    event_depth_model = baseline_model - pac_model
    max_depth_model = float(np.nanmax(event_depth_model)) if len(event_depth_model) else 0.0
    data_depth = baseline_model - mag_fit

    shoulder_mask = event_depth_model >= max(0.15 * max_depth_model, 0.03)
    left_mask = (jd_fit < t0) & shoulder_mask
    right_mask = (jd_fit > t0) & shoulder_mask
    shoulder_left = int(np.sum(left_mask))
    shoulder_right = int(np.sum(right_mask))
    strong_threshold = max(0.5 * max_depth_model, 0.04)
    n_strong_points = int(np.sum(data_depth >= strong_threshold))

    warnings = []
    fit_ok = True
    log10_delta_bic_vs_fred = _signed_log10_delta_bic(seed_result.get('delta_bic_vs_fred'))

    if not np.isfinite(seed_result.get('delta_bic_vs_flat', np.nan)) or float(seed_result['delta_bic_vs_flat']) < 10.0:
        warnings.append('weak_vs_flat')
        fit_ok = False
    if np.isfinite(log10_delta_bic_vs_fred) and log10_delta_bic_vs_fred <= -NON_PACZYNSKI_SELECTION_LOG10_DELTA_BIC_THRESHOLD:
        warnings.append('significant_fred_preferred')
        fit_ok = False
    if float(pac['reduced_chi2']) > 5.0:
        warnings.append('high_reduced_chi2')
        fit_ok = False
    if _finite_float(tE) is None or tE <= 0.0:
        warnings.append('invalid_tE')
        fit_ok = False
    t0_lower_gap = float(t0) - float(lower[1])
    t0_upper_gap = float(upper[1]) - float(t0)
    t0_margin = max(10.0, 0.02 * max(float(upper[1] - lower[1]), 1.0))
    if t0_lower_gap <= t0_margin or t0_upper_gap <= t0_margin:
        warnings.append('t0_near_bound')
        fit_ok = False

    tE_lower_limit = max(float(lower[2]) * 5.0, 5.0)
    tE_upper_margin = max(20.0, 0.10 * float(upper[2]))
    if float(tE) <= tE_lower_limit or float(upper[2]) - float(tE) <= tE_upper_margin:
        warnings.append('tE_near_bound')
        fit_ok = False

    A0_lower_limit = max(float(lower[0]) + 0.02, 1.02)
    A0_upper_margin = max(2.0, 0.05 * float(upper[0]))
    if float(A0) <= A0_lower_limit or float(upper[0]) - float(A0) <= A0_upper_margin:
        warnings.append('A0_near_bound')
        fit_ok = False
    if shoulder_left < 3 or shoulder_right < 3:
        warnings.append('insufficient_shoulders')
        fit_ok = False
    if n_strong_points < 2:
        warnings.append('single_point_peak')
        fit_ok = False

    return {
        'fit_ok': fit_ok,
        'warnings': warnings,
        'shoulder_left': shoulder_left,
        'shoulder_right': shoulder_right,
        'n_strong_points': n_strong_points,
        'max_model_depth': max_depth_model,
    }


def _best_selected_fit(seed_result: dict[str, object]) -> dict[str, object]:
    best_model = seed_result.get('selected_model') or seed_result.get('best_model')
    if not best_model:
        return {}
    return seed_result['fits'].get(best_model, {})


def _accepted_seed_rank(seed_result: dict[str, object]) -> tuple:
    pac = seed_result['fits'].get('paczynski', {})
    quality = seed_result.get('quality', {})
    return (
        int(bool(pac.get('success'))),
        int(seed_result.get('best_model') == 'paczynski'),
        int(bool(quality.get('fit_ok'))),
        float(seed_result.get('delta_bic_vs_fred', -1e9)) if np.isfinite(seed_result.get('delta_bic_vs_fred', np.nan)) else -1e9,
        float(seed_result.get('delta_bic_vs_flat', -1e9)) if np.isfinite(seed_result.get('delta_bic_vs_flat', np.nan)) else -1e9,
        -float(pac.get('reduced_chi2', np.inf)),
    )


def _significant_alt_model_name(seed_result: dict[str, object]) -> str | None:
    fred_fit = seed_result['fits'].get('fred', {})
    log10_delta_bic_vs_fred = _signed_log10_delta_bic(seed_result.get('delta_bic_vs_fred'))
    if fred_fit.get('success') and np.isfinite(log10_delta_bic_vs_fred) and log10_delta_bic_vs_fred <= -NON_PACZYNSKI_SELECTION_LOG10_DELTA_BIC_THRESHOLD:
        return 'fred'
    return None


def _significant_alt_seed_rank(seed_result: dict[str, object]) -> tuple:
    alt_model = _significant_alt_model_name(seed_result)
    alt_fit = seed_result['fits'].get(alt_model, {}) if alt_model else {}
    log10_delta_bic_vs_fred = _signed_log10_delta_bic(seed_result.get('delta_bic_vs_fred'))
    reduced_chi2 = float(alt_fit.get('reduced_chi2', np.inf))
    n_points = int(alt_fit.get('n_points', 0))
    return (
        int(bool(alt_fit.get('success'))),
        -float(log10_delta_bic_vs_fred) if np.isfinite(log10_delta_bic_vs_fred) else -1e9,
        -reduced_chi2 if np.isfinite(reduced_chi2) else -1e9,
        n_points,
    )


def _fallback_pacz_seed_rank(seed_result: dict[str, object]) -> tuple:
    pac = seed_result['fits'].get('paczynski', {})
    quality = seed_result.get('quality', {})
    reduced_chi2 = float(pac.get('reduced_chi2', np.inf))
    return (
        int(bool(pac.get('success'))),
        int(quality.get('shoulder_left', 0) + quality.get('shoulder_right', 0)),
        int(quality.get('n_strong_points', 0)),
        float(seed_result.get('delta_bic_vs_flat', -1e9)) if np.isfinite(seed_result.get('delta_bic_vs_flat', np.nan)) else -1e9,
        -reduced_chi2 if np.isfinite(reduced_chi2) else -1e9,
    )


def _fallback_seed_rank(seed_result: dict[str, object]) -> tuple:
    best_fit = _best_selected_fit(seed_result)
    best_model = seed_result.get('best_model')
    reduced_chi2 = float(best_fit.get('reduced_chi2', np.inf))
    n_points = int(best_fit.get('n_points', 0))
    support_score = (n_points / max(reduced_chi2, 0.5)) if np.isfinite(reduced_chi2) else -1.0
    return (
        int(bool(best_fit.get('success'))),
        float(support_score),
        -reduced_chi2 if np.isfinite(reduced_chi2) else -1e9,
        n_points,
        int(best_model not in (None, 'flat')),
        int(seed_result.get('quality', {}).get('shoulder_left', 0) + seed_result.get('quality', {}).get('shoulder_right', 0)),
    )


def _select_best_seed_result(seed_results: list[dict[str, object]]) -> tuple[dict[str, object], str]:
    accepted = [seed_result for seed_result in seed_results if seed_result.get('quality', {}).get('fit_ok')]
    if accepted:
        return max(accepted, key=_accepted_seed_rank), 'paczynski_qc'
    significant_alt = [seed_result for seed_result in seed_results if _significant_alt_model_name(seed_result)]
    if significant_alt:
        return max(significant_alt, key=_significant_alt_seed_rank), 'significant_alt_model'
    pac_fallback = [seed_result for seed_result in seed_results if seed_result['fits'].get('paczynski', {}).get('success')]
    if pac_fallback:
        return max(pac_fallback, key=_fallback_pacz_seed_rank), 'fallback_paczynski'
    return max(seed_results, key=_fallback_seed_rank), 'fallback_best_model'


def fit_candidate_context(context: dict[str, object]) -> dict[str, object]:
    df = context['df']
    row = context['row']
    width_seed = _pick_width_seed(row)
    seeds = _candidate_seeds(df, row)
    seed_results = []
    for seed in seeds:
        suite = _fit_model_suite(df, center_guess=float(seed['t0_guess']), width_seed=width_seed)
        suite['seed_method'] = str(seed['seed_method'])
        suite['seed_t0_guess'] = float(seed['t0_guess'])
        suite['quality'] = _pacz_quality_metrics(suite)
        seed_results.append(suite)

    best_seed_result, selection_mode = _select_best_seed_result(seed_results)
    quality = best_seed_result['quality']
    if selection_mode in ('paczynski_qc', 'fallback_paczynski') and best_seed_result['fits'].get('paczynski', {}).get('success'):
        selected_model_name = 'paczynski'
    elif selection_mode == 'significant_alt_model':
        selected_model_name = _significant_alt_model_name(best_seed_result)
    else:
        selected_model_name = str(best_seed_result.get('best_model')) if best_seed_result.get('best_model') is not None else None
    best_seed_result['selected_model'] = selected_model_name
    selected_fit = _best_selected_fit(best_seed_result)
    pac = best_seed_result['fits'].get('paczynski', {})
    raw_pacz_tE = float(abs(pac['params'][2])) if pac.get('success') else np.nan
    raw_pacz_t0 = float(pac['params'][1]) if pac.get('success') else np.nan
    pac_is_displayable = bool(pac.get('success')) and selected_model_name == 'paczynski'
    display_raw_tE = raw_pacz_tE if pac_is_displayable else np.nan
    display_raw_t0 = raw_pacz_t0 if pac_is_displayable else np.nan
    reported_tE = display_raw_tE if quality.get('fit_ok') else np.nan

    payload = context['payload']
    ra_deg = _finite_float(payload.get('ra_deg'))
    if ra_deg is None:
        ra_deg = _finite_float(payload.get('ra'))
    dec_deg = _finite_float(payload.get('dec_deg'))
    if dec_deg is None:
        dec_deg = _finite_float(payload.get('dec'))
    gaia_dr3_source_id = _text_value(payload.get('gaia_id'))
    vsx_name = _text_value(payload.get('vsx_name'))
    vsx_class = _text_value(payload.get('vsx_class'))
    vsx_sep_arcsec = _finite_float(payload.get('vsx_sep_arcsec'))
    asassn_var_name = _text_value(payload.get('asassn_var_name'))
    asassn_var_type = _text_value(payload.get('asassn_var_type'))
    simbad_sep_arcsec = _finite_float(payload.get('simbad_sep_arcsec'))

    fit_t0_jd_minus_2450000 = raw_pacz_t0 if np.isfinite(raw_pacz_t0) else np.nan
    fit_t0_jd = raw_pacz_t0 + 2450000.0 if np.isfinite(raw_pacz_t0) else np.nan
    peak_window_start_jd_minus_2450000 = raw_pacz_t0 - 2.0 * raw_pacz_tE if np.isfinite(raw_pacz_t0) and np.isfinite(raw_pacz_tE) else np.nan
    peak_window_end_jd_minus_2450000 = raw_pacz_t0 + 2.0 * raw_pacz_tE if np.isfinite(raw_pacz_t0) and np.isfinite(raw_pacz_tE) else np.nan
    peak_window_start_jd = peak_window_start_jd_minus_2450000 + 2450000.0 if np.isfinite(peak_window_start_jd_minus_2450000) else np.nan
    peak_window_end_jd = peak_window_end_jd_minus_2450000 + 2450000.0 if np.isfinite(peak_window_end_jd_minus_2450000) else np.nan

    if selected_model_name == 'paczynski':
        parallax_result = _fit_parallax_diagnostics(context, best_seed_result, ra_deg=ra_deg, dec_deg=dec_deg)
    else:
        parallax_result = _empty_parallax_result('not_attempted:selected_model_not_paczynski')

    brightest = df.nsmallest(min(5, len(df)), 'mag')
    summary = {
        'candidate_id': context['candidate_id'],
        'asas_sn_id': context['asas_sn_id'],
        'lc_path': str(context['lc_path']),
        'band_used': context['band_label'],
        'n_points_total': int(len(df)),
        'min_mag_t0_guess': float(df.loc[int(df['mag'].idxmin()), 'JD']),
        'brightest5_median_t0_guess': float(brightest['JD'].median()) if not brightest.empty else np.nan,
        'pipeline_jump_t0': _finite_float(row.get('jump_best_t0')),
        'seed_method': best_seed_result['seed_method'],
        'seed_t0_guess': best_seed_result['seed_t0_guess'],
        'best_model': selected_model_name,
        'best_model_by_bic': best_seed_result.get('best_model'),
        'best_alt_model': best_seed_result.get('best_alt_model'),
        'selection_mode': selection_mode,
        'fit_ok': bool(quality.get('fit_ok')),
        'reported_tE_days': reported_tE,
        'raw_paczynski_tE_days': raw_pacz_tE,
        'display_raw_paczynski_tE_days': display_raw_tE,
        'fit_t0': display_raw_t0,
        'fit_t0_time_system': 'JD-2450000',
        'fit_t0_jd_minus_2450000': fit_t0_jd_minus_2450000,
        'fit_t0_jd': fit_t0_jd,
        'peak_window_time_system': 'JD-2450000',
        'peak_window_start_jd_minus_2450000': peak_window_start_jd_minus_2450000,
        'peak_window_end_jd_minus_2450000': peak_window_end_jd_minus_2450000,
        'peak_window_start_jd': peak_window_start_jd,
        'peak_window_end_jd': peak_window_end_jd,
        'fit_reduced_chi2': float(selected_fit.get('reduced_chi2', np.nan)),
        'delta_bic_vs_flat': best_seed_result.get('delta_bic_vs_flat'),
        'delta_bic_vs_gaussian': best_seed_result.get('delta_bic_vs_gaussian'),
        'delta_bic_vs_fred': best_seed_result.get('delta_bic_vs_fred'),
        'delta_bic_vs_best_alt': best_seed_result.get('delta_bic_vs_best_alt'),
        'n_points_fit': int(np.sum(best_seed_result['fit_mask'])),
        'half_window_days': float(best_seed_result['half_window']),
        'shoulder_left': int(quality.get('shoulder_left', 0)),
        'shoulder_right': int(quality.get('shoulder_right', 0)),
        'n_strong_points': int(quality.get('n_strong_points', 0)),
        'fit_warning': ','.join(quality.get('warnings', [])),
        'ra_deg': ra_deg,
        'dec_deg': dec_deg,
        'gaia_dr3_source_id': gaia_dr3_source_id,
        'asassn_source_id': _text_value(context['asas_sn_id']),
        'asassn_var_name': asassn_var_name,
        'asassn_var_type': asassn_var_type,
        'nearest_simbad_object': _text_value(row.get('simbad_main_id')) or _text_value(payload.get('simbad_main_id')),
        'simbad_otype': _text_value(row.get('simbad_otype')) or _text_value(payload.get('simbad_otype')),
        'simbad_sep_arcsec': simbad_sep_arcsec,
        'nearest_vsx_object': vsx_name,
        'vsx_class': vsx_class,
        'vsx_sep_arcsec': vsx_sep_arcsec,
        'microlens_match': _bool_flag(row.get('microlens_match')),
        'microlens_catalog': _text_value(row.get('microlens_catalog')),
        'microlens_name': _text_value(row.get('microlens_name')),
        'microlens_alt_name': _text_value(row.get('microlens_alt_name')),
        'microlens_te_days': _finite_float(row.get('microlens_te_days')),
        'microlens_sep_arcsec': _finite_float(row.get('microlens_sep_arcsec')),
        'vetting_likely_known': _bool_flag(row.get('vetting_likely_known')),
        'catalog_source': _text_value(row.get('catalog_source')),
        'gaia_var_class': _text_value(payload.get('gaia_var_class')) or _text_value(row.get('gaia_var_class')),
        'ztf_var_type': _text_value(payload.get('ztf_var_type')) or _text_value(row.get('ztf_var_type')),
    }
    summary.update(_flatten_parallax_summary(parallax_result))

    return {
        'context': context,
        'seed_results': seed_results,
        'best_seed_result': best_seed_result,
        'parallax': parallax_result,
        'summary': summary,
    }

def fit_march18_candidates(
    db_path: str | Path,
    *,
    candidate_ids: list[str],
    prefer_g_band: bool = True,
) -> tuple[pd.DataFrame, list[dict[str, object]]]:
    db_path = Path(db_path).expanduser().resolve()
    plot_dir = infer_plot_dir_from_source(db_path)
    results = []
    with sqlite3.connect(db_path) as conn:
        for candidate_id in candidate_ids:
            context = _load_candidate_context(
                conn,
                str(candidate_id),
                plot_dir=plot_dir,
                prefer_g_band=prefer_g_band,
            )
            results.append(fit_candidate_context(context))

    results_df = pd.DataFrame([result['summary'] for result in results]).sort_values('candidate_id').reset_index(drop=True)
    return results_df, results


def plot_candidate_fit(result: dict[str, object], *, figsize: tuple[float, float] = (13.0, 8.5)):
    context = result['context']
    summary = result['summary']
    best_seed = result['best_seed_result']
    quality = best_seed['quality']
    pac = best_seed['fits'].get('paczynski', {})
    best_model_name = best_seed.get('selected_model') or best_seed.get('best_model')
    best_model_fit = best_seed['fits'].get(best_model_name, {}) if best_model_name else {}
    show_paczynski = bool(pac.get('success')) and best_model_name == 'paczynski'
    df = context['df']
    parallax = result.get('parallax', {}) or {}
    parallax_best = parallax.get('branches', {}).get(parallax.get('best_branch', ''), {}) if parallax.get('best_branch') else {}
    show_parallax = False # bool(parallax_best.get('success')) and summary.get('parallax_attempted', False)

    plot_jd_offset = 8000.0
    jd_axis_label = 'JD - 2458000 [d]'
    mag_label = r'$g$ [mag]'

    def _plot_jd(values):
        return np.asarray(values, dtype=float) - plot_jd_offset

    def _plot_jd_scalar(value: float) -> float:
        return float(value) - plot_jd_offset

    fig = plt.figure(figsize=figsize, dpi=220)
    gs = fig.add_gridspec(3, 1, height_ratios=[3.2, 2.1, 1.0], hspace=0.12)
    ax = fig.add_subplot(gs[0])
    ax_zoom = fig.add_subplot(gs[1])
    ax_res = fig.add_subplot(gs[2], sharex=ax_zoom)
    fig.subplots_adjust(left=0.08, right=0.78, top=0.93, bottom=0.10)

    fit_mask = np.asarray(best_seed['fit_mask'], dtype=bool)
    fit_df = df.loc[fit_mask]
    jd_dense = np.linspace(float(best_seed['jd_fit'].min()), float(best_seed['jd_fit'].max()), 700)
    dense_models = {}
    if pac.get('success'):
        dense_models['paczynski'] = _evaluate_model('paczynski', pac['params'], jd_dense, pac['t_ref'])
    best_chi2 = best_model_fit.get('chi2', float('inf'))
    for m_name in ['gaussian', 'fred']:
        m_fit = best_seed['fits'].get(m_name, {})
        m_chi2 = m_fit.get('chi2')
        if m_fit.get('success') and m_chi2 is not None and (m_chi2 - best_chi2) <= -25:
            dense_models[m_name] = _evaluate_model(m_name, m_fit['params'], jd_dense, m_fit['t_ref'])
    parallax_dense = None
    if show_parallax and np.isfinite(summary.get('ra_deg', np.nan)) and np.isfinite(summary.get('dec_deg', np.nan)):
        parallax_dense = _evaluate_parallax_branch_mag(parallax_best, jd_dense, float(summary['ra_deg']), float(summary['dec_deg']))

    def _zoom_center_and_scale() -> tuple[float, float]:
        if show_parallax:
            return float(summary['parallax_best_t0_jd_minus_2450000']), max(float(summary['parallax_best_tE_days']), 5.0)
        if best_model_fit.get('success') and best_model_name in {'paczynski', 'gaussian', 'fred'}:
            params = np.asarray(best_model_fit['params'], dtype=float)
            center = float(params[1])
            if best_model_name in {'paczynski', 'gaussian'}:
                scale = float(abs(params[2]))
            else:
                scale = float(max(abs(params[2]), abs(params[3])))
            return center, max(scale, 5.0)
        if pac.get('success'):
            return float(pac['params'][1]), max(float(abs(pac['params'][2])), 5.0)
        return float(summary['seed_t0_guess']), max(float(0.12 * best_seed['half_window']), 10.0)

    zoom_center, zoom_scale = _zoom_center_and_scale()
    zoom_half_window = float(np.clip(max(35.0, 3.5 * zoom_scale), 35.0, max(80.0, float(best_seed['half_window']))))
    zoom_mask = np.abs(df['JD'] - zoom_center) <= zoom_half_window
    if int(np.sum(zoom_mask)) < 8:
        zoom_half_window = float(np.clip(max(60.0, 0.25 * float(best_seed['half_window'])), 60.0, max(120.0, float(best_seed['half_window']))))
        zoom_mask = np.abs(df['JD'] - zoom_center) <= zoom_half_window
    zoom_df = df.loc[zoom_mask]
    zoom_fit_df = fit_df.loc[np.abs(fit_df['JD'] - zoom_center) <= zoom_half_window] if not fit_df.empty else fit_df

    for axis in (ax, ax_zoom):
        axis.errorbar(_plot_jd(df['JD']), df['mag'], yerr=df['mag_err'], fmt='k.', alpha=0.25 if axis is ax_zoom else 0.7, markersize=3, elinewidth=0.8, capsize=0)
        axis.errorbar(_plot_jd(fit_df['JD']), fit_df['mag'], yerr=fit_df['mag_err'], fmt='k.', alpha=0.9, markersize=4, elinewidth=1.0, capsize=0)
        colors = {'paczynski': 'red', 'fred': 'blue', 'gaussian': 'orange'}
        for m_name, m_dense in dense_models.items():
            kw = {'color': colors.get(m_name, 'k'), 'linewidth': 2.0 if m_name == 'paczynski' else 1.6}
            if m_name != 'paczynski': kw['linestyle'] = '--'
            if axis is ax: kw['label'] = f'{m_name.capitalize()} fit'
            axis.plot(_plot_jd(jd_dense), m_dense, **kw)
        if show_paczynski:
            axis.axvline(_plot_jd_scalar(float(pac['params'][1])), color='tab:orange', linestyle=':', linewidth=1.2, label='Pacz fit t0' if axis is ax else None)
        if parallax_dense is not None:
            axis.plot(_plot_jd(jd_dense), parallax_dense, color='tab:cyan', linewidth=2.0, linestyle='-.')
        axis.axvline(_plot_jd_scalar(float(summary['seed_t0_guess'])), color='tab:brown', linestyle='--', linewidth=1.0, alpha=0.8)
        axis.grid(alpha=0.2)
        axis.invert_yaxis()

    for artist in [*list(ax_zoom.collections), *list(ax_zoom.lines)]:
        artist.remove()
    ax_zoom.errorbar(_plot_jd(zoom_df['JD']), zoom_df['mag'], yerr=zoom_df['mag_err'], fmt='k.', alpha=0.8, markersize=4, elinewidth=1.0, capsize=0)
    ax_zoom.errorbar(_plot_jd(zoom_fit_df['JD']), zoom_fit_df['mag'], yerr=zoom_fit_df['mag_err'], fmt='k.', alpha=0.95, markersize=5, elinewidth=1.2, capsize=0)
    zoom_dense_mask = np.abs(jd_dense - zoom_center) <= zoom_half_window
    colors = {'paczynski': 'red', 'fred': 'blue', 'gaussian': 'orange'}
    for m_name, m_dense in dense_models.items():
        kw = {'color': colors.get(m_name, 'k'), 'linewidth': 2.0 if m_name == 'paczynski' else 1.6}
        if m_name != 'paczynski': kw['linestyle'] = '--'
        ax_zoom.plot(_plot_jd(jd_dense[zoom_dense_mask]), m_dense[zoom_dense_mask], **kw)
    if parallax_dense is not None:
        ax_zoom.plot(_plot_jd(jd_dense[zoom_dense_mask]), parallax_dense[zoom_dense_mask], color='tab:cyan', linewidth=2.0, linestyle='-.')
    ax_zoom.axvline(_plot_jd_scalar(float(summary['seed_t0_guess'])), color='tab:brown', linestyle='--', linewidth=1.0, alpha=0.8)
    if show_paczynski:
        ax_zoom.axvline(_plot_jd_scalar(float(pac['params'][1])), color='tab:orange', linestyle=':', linewidth=1.2)
    if show_parallax and np.isfinite(summary.get('parallax_best_t0_jd_minus_2450000', np.nan)):
        ax_zoom.axvline(_plot_jd_scalar(float(summary['parallax_best_t0_jd_minus_2450000'])), color='tab:cyan', linestyle=':', linewidth=1.2)
    ax_zoom.set_xlim(_plot_jd_scalar(zoom_center - zoom_half_window), _plot_jd_scalar(zoom_center + zoom_half_window))
    ax_zoom.set_ylabel(mag_label)
    ax_zoom.tick_params(axis='x', which='both', labelbottom=False, labeltop=True, top=True)

    if show_parallax:
        residual_model = np.asarray(parallax_best.get('model_mag', np.full_like(best_seed['mag_fit'], np.nan)), dtype=float)
        residual_label = 'Parallax residual'
    elif best_model_fit.get('success'):
        residual_model = best_model_fit['model']
        residual_label = 'Pacz residual' if best_model_name == 'paczynski' else f"{best_model_name} residual"
    else:
        residual_model = np.full_like(best_seed['mag_fit'], np.nan)
        residual_label = 'Residual'

    residuals = best_seed['mag_fit'] - residual_model
    zoom_res_mask = np.abs(best_seed['jd_fit'] - zoom_center) <= zoom_half_window
    if int(np.sum(zoom_res_mask)) < 4:
        zoom_res_mask = np.ones_like(best_seed['jd_fit'], dtype=bool)
    ax_res.axhline(0.0, color='0.4', linewidth=1.0)
    ax_res.errorbar(_plot_jd(best_seed['jd_fit'][zoom_res_mask]), residuals[zoom_res_mask], yerr=best_seed.get('mag_err_fit', [None]*len(best_seed['jd_fit']))[zoom_res_mask] if 'mag_err_fit' in best_seed else None, fmt='k.', alpha=0.9, markersize=4, elinewidth=1.0, capsize=0, label=residual_label)
    ax_res.set_ylabel('Residual [mag]')
    ax_res.set_xlabel(jd_axis_label)
    ax_res.grid(alpha=0.2)
    ax_res.invert_yaxis()

    title_tE_days = summary['reported_tE_days']
    if not np.isfinite(title_tE_days):
        title_tE_days = summary['raw_paczynski_tE_days']
    title = f"{summary['candidate_id']} | tE={title_tE_days if np.isfinite(title_tE_days) else float('nan'):.3f} d"
    ax.set_title(title)
    ax.set_ylabel(mag_label)

    best_chi2 = best_model_fit.get('chi2', np.nan)
    dbic_lines = []
    for m_name in ['flat', 'gaussian', 'fred', 'paczynski']:
        if m_name == best_model_name: continue
        m_chi2 = best_seed['fits'].get(m_name, {}).get('chi2')
        if m_chi2 is not None and np.isfinite(m_chi2) and np.isfinite(best_chi2):
            dchi2_val = m_chi2 - best_chi2
            dbic_lines.append(f"dChi2({m_name})={dchi2_val:.2f}")
    par_chi2 = summary.get('parallax_best_chi2')
    if par_chi2 is not None and np.isfinite(par_chi2) and np.isfinite(best_chi2) and best_model_name != 'parallax':
        dchi2_val = par_chi2 - best_chi2
        dbic_lines.append(f"dChi2(parallax)={dchi2_val:.2f}")
    info_lines = [
        f"best={summary['best_model']}",
        f"chi2_nu={summary['fit_reduced_chi2']:.2f}",
        *dbic_lines,
    ]
    if summary.get('parallax_attempted', False):
        par_status = summary.get('parallax_status', 'attempted')
        dchi2 = summary.get('parallax_delta_chi2')
        dbic = summary.get('parallax_delta_bic')
        par_line = f"parallax={par_status}"
        if dchi2 is not None and np.isfinite(dchi2):
            par_line += f" | dchi2={float(dchi2):.2f}"
        info_lines.append(par_line)
        if np.isfinite(summary.get('parallax_best_piE', np.nan)):
            info_lines.append(
                f"best branch={summary.get('parallax_best_branch', '')} | piE={float(summary['parallax_best_piE']):.3f} | u0={float(summary['parallax_best_u0']):.4f}"
            )
    from matplotlib.offsetbox import AnchoredText
    for axis in (ax, ax_zoom, ax_res):
        at = AnchoredText('
'.join(info_lines), loc='best', prop=dict(fontsize=8, alpha=0.9), frameon=True)
        at.patch.set_boxstyle('round,pad=0.35')
        axis.add_artist(at)
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(handles, labels, loc='upper right', fontsize=8)
    ax_res.legend(loc='best', fontsize=8)
    return fig, (ax, ax_zoom, ax_res)



In [4]:
results_df, fit_results = fit_march18_candidates(
    DB_PATH,
    candidate_ids=MARCH18_CANDIDATE_IDS,
    prefer_g_band=True,
)

import io
from pathlib import Path

import astropy.units as u
from astropy.coordinates import SkyCoord

from malca.vetting import (
    MICROLENS_CACHE_DIR,
    fetch_microlensing_event_catalog,
    _safe_text,
)

EXTERNAL_MICROLENS_RADIUS_ARCSEC = 2.0
GAIA_ALERT_RADIUS_ARCSEC = 2.0
REFRESH_EXTERNAL_MICROLENS_TABLES = False
RUN_GAIA_ALERT_LOOKUP = True
MANUAL_OGLE_EWS_CSV_CANDIDATES = [
    REPO_ROOT / 'input' / 'ogle-ews-220326.csv',
    REPO_ROOT / 'input' / 'ogle_ews_220326.csv',
]
MANUAL_OGLE_EWS_CSV_PATH = next((path for path in MANUAL_OGLE_EWS_CSV_CANDIDATES if path.exists()), MANUAL_OGLE_EWS_CSV_CANDIDATES[-1])
MICROLENS_SOURCE_PREFIX = {
    'OGLE-EWS-220326': 'ogle_ews_220326',
    'OGLE-EWS': 'ogle_ews',
    'KMTNet': 'kmtnet',
    'MOA': 'moa',
}
MICROLENS_SOURCE_PRIORITY = {
    'OGLE-EWS-220326': 0,
    'OGLE-EWS': 1,
    'KMTNet': 2,
    'MOA': 3,
}
EXTERNAL_MATCH_DETAIL_COLS = [
    'candidate_id',
    'source',
    'event_id',
    'alias',
    'sep_arcsec',
    'catalog_t0_time_system',
    'catalog_t0_jd',
    'delta_t0_days',
    'catalog_tE_days',
    'catalog_tE_kind',
    'delta_tE_days',
    'delta_log_tE',
    'status',
    'event_year',
    'source_url',
    'match_rank',
    'source_match_rank',
]


def signed_log10_series(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors='coerce')
    return np.sign(values) * np.log10(1.0 + np.abs(values))


def _external_cache_path(name: str) -> Path:
    root = Path(MICROLENS_CACHE_DIR).expanduser()
    root.mkdir(parents=True, exist_ok=True)
    return root / name


def _empty_external_match_summary(candidate_ids: pd.Series | list[str]) -> pd.DataFrame:
    candidate_ids = [str(cid) for cid in candidate_ids]
    df = pd.DataFrame({'candidate_id': candidate_ids})
    defaults: dict[str, object] = {
        'external_known_microlens': False,
        'external_best_microlens_source': '',
        'external_best_microlens_event_id': '',
        'external_best_microlens_alias': '',
        'external_best_microlens_sep_arcsec': np.nan,
        'external_best_microlens_catalog_t0_jd': np.nan,
        'external_best_microlens_catalog_t0_time_system': '',
        'external_best_microlens_delta_t0_days': np.nan,
        'external_best_microlens_catalog_tE_days': np.nan,
        'external_best_microlens_catalog_tE_kind': '',
        'external_best_microlens_delta_tE_days': np.nan,
        'external_best_microlens_delta_log_tE': np.nan,
        'external_best_microlens_status': '',
        'external_best_microlens_event_year': np.nan,
        'external_best_microlens_source_url': '',
    }
    per_source_defaults: dict[str, object] = {}
    for prefix in MICROLENS_SOURCE_PREFIX.values():
        per_source_defaults.update({
            f'{prefix}_event_id': '',
            f'{prefix}_alias': '',
            f'{prefix}_sep_arcsec': np.nan,
            f'{prefix}_catalog_t0_jd': np.nan,
            f'{prefix}_catalog_t0_time_system': '',
            f'{prefix}_delta_t0_days': np.nan,
            f'{prefix}_catalog_tE_days': np.nan,
            f'{prefix}_catalog_tE_kind': '',
            f'{prefix}_delta_tE_days': np.nan,
            f'{prefix}_delta_log_tE': np.nan,
            f'{prefix}_status': '',
            f'{prefix}_source_url': '',
        })
    for col, default in {**defaults, **per_source_defaults}.items():
        df[col] = default
    return df


def _empty_gaia_alert_summary(candidate_ids: pd.Series | list[str]) -> pd.DataFrame:
    candidate_ids = [str(cid) for cid in candidate_ids]
    df = pd.DataFrame({'candidate_id': candidate_ids})
    df['gaia_alert_name'] = ''
    df['gaia_alert_class'] = ''
    df['gaia_alert_sep_arcsec'] = np.nan
    df['gaia_alert_microlens_like'] = False
    df['gaia_alert_lookup_status'] = 'not_run'
    return df


def _normalize_event_id(event_raw: object, prefix: str) -> str:
    text = _safe_text(event_raw)
    if not text:
        return ''
    if text.upper().startswith(prefix.upper() + '-'):
        return text
    return f'{prefix}-{text}'










def build_external_microlens_catalog(*, force_refresh: bool = False) -> pd.DataFrame:
    return fetch_microlensing_event_catalog(show_tqdm=True)


def _abs_diff(left: pd.Series, right: pd.Series) -> pd.Series:
    left = pd.to_numeric(left, errors='coerce')
    right = pd.to_numeric(right, errors='coerce')
    return (left - right).abs()


def _delta_log_te(left: pd.Series, right: pd.Series) -> pd.Series:
    left = pd.to_numeric(left, errors='coerce')
    right = pd.to_numeric(right, errors='coerce')
    valid = (left > 0.0) & (right > 0.0)
    out = pd.Series(np.nan, index=left.index, dtype=float)
    out.loc[valid] = np.abs(np.log10(left.loc[valid] / right.loc[valid]))
    return out


def crossmatch_external_microlensing(master_table: pd.DataFrame, *, radius_arcsec: float, force_refresh: bool = False) -> tuple[pd.DataFrame, pd.DataFrame]:
    summary_df = _empty_external_match_summary(master_table['candidate_id'])
    catalog = build_external_microlens_catalog(force_refresh=force_refresh)
    if catalog.empty:
        return summary_df, pd.DataFrame(columns=EXTERNAL_MATCH_DETAIL_COLS)

    candidate_cols = ['candidate_id', 'ra_deg', 'dec_deg', 'fit_t0_jd', 'raw_paczynski_tE_days']
    candidate_table = master_table[candidate_cols].copy()
    valid_candidates = candidate_table['ra_deg'].notna() & candidate_table['dec_deg'].notna()
    valid_catalog = catalog['ra'].notna() & catalog['dec'].notna()
    if not valid_candidates.any() or not valid_catalog.any():
        return summary_df, pd.DataFrame(columns=EXTERNAL_MATCH_DETAIL_COLS)

    cand_valid = candidate_table.loc[valid_candidates].reset_index(drop=True)
    cat_valid = catalog.loc[valid_catalog].reset_index(drop=True)

    cand_coords = SkyCoord(ra=cand_valid['ra_deg'].to_numpy(dtype=float), dec=cand_valid['dec_deg'].to_numpy(dtype=float), unit='deg')
    cat_coords = SkyCoord(ra=cat_valid['ra'].to_numpy(dtype=float), dec=cat_valid['dec'].to_numpy(dtype=float), unit='deg')
    idx_cat, idx_cand, sep2d, _ = cand_coords.search_around_sky(cat_coords, radius_arcsec * u.arcsec)
    if len(idx_cand) == 0:
        return summary_df, pd.DataFrame(columns=EXTERNAL_MATCH_DETAIL_COLS)

    matches = cat_valid.iloc[np.asarray(idx_cat, dtype=int)].copy().reset_index(drop=True)
    matched_candidates = cand_valid.iloc[np.asarray(idx_cand, dtype=int)].reset_index(drop=True)
    matches['candidate_id'] = matched_candidates['candidate_id'].astype(str)
    matches['candidate_fit_t0_jd'] = matched_candidates['fit_t0_jd'].to_numpy(dtype=float)
    matches['candidate_fit_tE_days'] = matched_candidates['raw_paczynski_tE_days'].to_numpy(dtype=float)
    matches['sep_arcsec'] = np.asarray(sep2d.arcsec, dtype=float)
    matches['delta_t0_days'] = _abs_diff(matches['candidate_fit_t0_jd'], matches['catalog_t0_jd'])
    matches['delta_tE_days'] = _abs_diff(matches['candidate_fit_tE_days'], matches['catalog_tE_days'])
    matches['delta_log_tE'] = _delta_log_te(matches['candidate_fit_tE_days'], matches['catalog_tE_days'])
    matches['source_rank'] = matches['source'].map(MICROLENS_SOURCE_PRIORITY).fillna(999).astype(int)
    matches['t0_missing_rank'] = matches['delta_t0_days'].isna().astype(int)
    matches['tE_missing_rank'] = matches['delta_log_tE'].isna().astype(int)
    matches = matches.sort_values(
        ['candidate_id', 'sep_arcsec', 't0_missing_rank', 'delta_t0_days', 'tE_missing_rank', 'delta_log_tE', 'source_rank', 'event_id'],
        na_position='last',
    ).reset_index(drop=True)
    matches['match_rank'] = matches.groupby('candidate_id').cumcount() + 1
    matches['source_match_rank'] = matches.groupby(['candidate_id', 'source']).cumcount() + 1

    summary_rows: list[dict[str, object]] = []
    for candidate_id, group in matches.groupby('candidate_id', sort=False):
        row = _empty_external_match_summary([candidate_id]).iloc[0].to_dict()
        row['candidate_id'] = str(candidate_id)
        row['external_known_microlens'] = True
        best = group.iloc[0]
        row.update({
            'external_best_microlens_source': _safe_text(best.get('source')),
            'external_best_microlens_event_id': _safe_text(best.get('event_id')),
            'external_best_microlens_alias': _safe_text(best.get('alias')),
            'external_best_microlens_sep_arcsec': float(best.get('sep_arcsec')) if pd.notna(best.get('sep_arcsec')) else np.nan,
            'external_best_microlens_catalog_t0_jd': float(best.get('catalog_t0_jd')) if pd.notna(best.get('catalog_t0_jd')) else np.nan,
            'external_best_microlens_catalog_t0_time_system': _safe_text(best.get('catalog_t0_time_system')),
            'external_best_microlens_delta_t0_days': float(best.get('delta_t0_days')) if pd.notna(best.get('delta_t0_days')) else np.nan,
            'external_best_microlens_catalog_tE_days': float(best.get('catalog_tE_days')) if pd.notna(best.get('catalog_tE_days')) else np.nan,
            'external_best_microlens_catalog_tE_kind': _safe_text(best.get('catalog_tE_kind')),
            'external_best_microlens_delta_tE_days': float(best.get('delta_tE_days')) if pd.notna(best.get('delta_tE_days')) else np.nan,
            'external_best_microlens_delta_log_tE': float(best.get('delta_log_tE')) if pd.notna(best.get('delta_log_tE')) else np.nan,
            'external_best_microlens_status': _safe_text(best.get('status')),
            'external_best_microlens_event_year': float(best.get('event_year')) if pd.notna(best.get('event_year')) else np.nan,
            'external_best_microlens_source_url': _safe_text(best.get('source_url')),
        })
        for source_name, prefix in MICROLENS_SOURCE_PREFIX.items():
            source_group = group.loc[group['source'] == source_name]
            if source_group.empty:
                continue
            match = source_group.iloc[0]
            row.update({
                f'{prefix}_event_id': _safe_text(match.get('event_id')),
                f'{prefix}_alias': _safe_text(match.get('alias')),
                f'{prefix}_sep_arcsec': float(match.get('sep_arcsec')) if pd.notna(match.get('sep_arcsec')) else np.nan,
                f'{prefix}_catalog_t0_jd': float(match.get('catalog_t0_jd')) if pd.notna(match.get('catalog_t0_jd')) else np.nan,
                f'{prefix}_catalog_t0_time_system': _safe_text(match.get('catalog_t0_time_system')),
                f'{prefix}_delta_t0_days': float(match.get('delta_t0_days')) if pd.notna(match.get('delta_t0_days')) else np.nan,
                f'{prefix}_catalog_tE_days': float(match.get('catalog_tE_days')) if pd.notna(match.get('catalog_tE_days')) else np.nan,
                f'{prefix}_catalog_tE_kind': _safe_text(match.get('catalog_tE_kind')),
                f'{prefix}_delta_tE_days': float(match.get('delta_tE_days')) if pd.notna(match.get('delta_tE_days')) else np.nan,
                f'{prefix}_delta_log_tE': float(match.get('delta_log_tE')) if pd.notna(match.get('delta_log_tE')) else np.nan,
                f'{prefix}_status': _safe_text(match.get('status')),
                f'{prefix}_source_url': _safe_text(match.get('source_url')),
            })
        summary_rows.append(row)

    summary_df = _empty_external_match_summary(master_table['candidate_id']).merge(
        pd.DataFrame(summary_rows),
        on='candidate_id',
        how='left',
        suffixes=('', '_match'),
    )
    for col in summary_df.columns:
        if col.endswith('_match'):
            base_col = col[:-6]
            summary_df[base_col] = summary_df[col].combine_first(summary_df[base_col])
            summary_df = summary_df.drop(columns=[col])

    detail_df = matches[EXTERNAL_MATCH_DETAIL_COLS].copy()
    return summary_df, detail_df


def crossmatch_external_gaia_alerts(master_table: pd.DataFrame, *, radius_arcsec: float, run_lookup: bool) -> pd.DataFrame:
    summary_df = _empty_gaia_alert_summary(master_table['candidate_id'])
    if not run_lookup:
        summary_df['gaia_alert_lookup_status'] = 'skipped'
        return summary_df
    try:
        from malca.ltv.crossmatch import crossmatch_gaia_alerts
    except Exception as exc:
        summary_df['gaia_alert_lookup_status'] = f'import_failed:{type(exc).__name__}'
        return summary_df

    try:
        gaia_df = crossmatch_gaia_alerts(
            master_table[['candidate_id', 'ra_deg', 'dec_deg']].copy(),
            ra_column='ra_deg',
            dec_column='dec_deg',
            match_radius_arcsec=radius_arcsec,
            verbose=False,
        )
    except Exception as exc:
        summary_df['gaia_alert_lookup_status'] = f'query_failed:{type(exc).__name__}'
        return summary_df

    out = summary_df.merge(
        gaia_df[['candidate_id', 'gaia_alert_name', 'gaia_alert_class', 'gaia_alert_sep_arcsec']].copy(),
        on='candidate_id',
        how='left',
        suffixes=('', '_new'),
    )
    for col in ('gaia_alert_name', 'gaia_alert_class', 'gaia_alert_sep_arcsec'):
        out[col] = out[f'{col}_new'].combine_first(out[col])
        out = out.drop(columns=[f'{col}_new'])
    out['gaia_alert_name'] = out['gaia_alert_name'].fillna('')
    out['gaia_alert_class'] = out['gaia_alert_class'].fillna('')
    out['gaia_alert_microlens_like'] = out['gaia_alert_class'].astype(str).str.contains(r'ULENS|MICROLENS|LENS', case=False, regex=True, na=False)
    out['gaia_alert_lookup_status'] = 'queried'
    return out


external_summary_df, external_match_details_df = crossmatch_external_microlensing(
    results_df,
    radius_arcsec=EXTERNAL_MICROLENS_RADIUS_ARCSEC,
    force_refresh=REFRESH_EXTERNAL_MICROLENS_TABLES,
)
gaia_alert_summary_df = crossmatch_external_gaia_alerts(
    results_df,
    radius_arcsec=GAIA_ALERT_RADIUS_ARCSEC,
    run_lookup=RUN_GAIA_ALERT_LOOKUP,
)

master_df = results_df.copy()
master_df = master_df.merge(external_summary_df, on='candidate_id', how='left')
master_df = master_df.merge(gaia_alert_summary_df, on='candidate_id', how='left')


asassn_ml_path = REPO_ROOT / 'input' / 'asas_sn_microlens.csv'
master_df['asassn_ml_match'] = False
master_df['asassn_ml_name'] = ''
master_df['asassn_ml_sep_arcsec'] = np.nan
if asassn_ml_path.exists():
    asml = pd.read_csv(asassn_ml_path, header=None)
    asml_ra = pd.to_numeric(asml.iloc[:, 12], errors='coerce')
    asml_dec = pd.to_numeric(asml.iloc[:, 13], errors='coerce')
    asml_name = asml.iloc[:, 0].replace('', pd.NA).combine_first(asml.iloc[:, 1]).fillna('unknown')
    valid = asml_ra.notna() & asml_dec.notna()
    if valid.any():
        cat_coords = SkyCoord(ra=asml_ra[valid].values, dec=asml_dec[valid].values, unit='deg')
        asml_names = asml_name[valid].values
        
        master_coords = SkyCoord(ra=master_df['ra_deg'].values, dec=master_df['dec_deg'].values, unit='deg')
        idx_cat, sep2d, _ = master_coords.match_to_catalog_sky(cat_coords)
        match_mask = sep2d <= 5.0 * u.arcsec
        
        if len(idx_cat) > 0:
            master_df.loc[match_mask, 'asassn_ml_match'] = True
            master_df.loc[match_mask, 'asassn_ml_name'] = asml_names[idx_cat][match_mask]
            master_df.loc[match_mask, 'asassn_ml_sep_arcsec'] = sep2d.arcsec[match_mask]
master_df['external_known_microlens'] = master_df['external_known_microlens'].fillna(False).astype(bool)
master_df['gaia_alert_microlens_like'] = master_df['gaia_alert_microlens_like'].fillna(False).astype(bool)
master_df['external_known_microlens'] = master_df['external_known_microlens'] | master_df['gaia_alert_microlens_like'] | master_df['asassn_ml_match']
master_df['log10_delta_bic_vs_flat'] = signed_log10_series(master_df['delta_bic_vs_flat'])
master_df['log10_delta_bic_vs_gaussian'] = signed_log10_series(master_df['delta_bic_vs_gaussian'])
master_df['log10_delta_bic_vs_fred'] = signed_log10_series(master_df['delta_bic_vs_fred'])
master_df['log10_delta_bic_vs_best_alt'] = signed_log10_series(master_df['delta_bic_vs_best_alt'])

summary_lookup_cols = [col for col in master_df.columns if col not in results_df.columns]
summary_lookup = master_df.set_index('candidate_id')[summary_lookup_cols].to_dict('index')
for result in fit_results:
    result['summary'].update(summary_lookup.get(result['summary']['candidate_id'], {}))

master_cols = [
    'candidate_id',
    'asassn_source_id',
    'asassn_var_name',
    'asassn_var_type',
    'gaia_dr3_source_id',
    'ra_deg',
    'dec_deg',
    'fit_t0_time_system',
    'fit_t0_jd_minus_2450000',
    'fit_t0_jd',
    'raw_paczynski_tE_days',
    'parallax_attempted',
    'parallax_fit_ok',
    'parallax_preferred',
    'parallax_status',
    'parallax_warning',
    'parallax_best_branch',
    'parallax_delta_chi2',
    'parallax_delta_bic',
    'parallax_best_t0_jd_minus_2450000',
    'parallax_best_tE_days',
    'parallax_best_u0',
    'parallax_best_piE_N',
    'parallax_best_piE_E',
    'parallax_best_piE',
    'parallax_best_chi2',
    'parallax_best_reduced_chi2',
    'parallax_best_bic',
    'parallax_best_acceptance_rate',
    'parallax_best_n_samples',
    'parallax_pos_t0_jd_minus_2450000',
    'parallax_pos_tE_days',
    'parallax_pos_u0',
    'parallax_pos_piE_N',
    'parallax_pos_piE_E',
    'parallax_pos_piE',
    'parallax_pos_chi2',
    'parallax_neg_t0_jd_minus_2450000',
    'parallax_neg_tE_days',
    'parallax_neg_u0',
    'parallax_neg_piE_N',
    'parallax_neg_piE_E',
    'parallax_neg_piE',
    'parallax_neg_chi2',
    'peak_window_time_system',
    'peak_window_start_jd_minus_2450000',
    'peak_window_end_jd_minus_2450000',
    'peak_window_start_jd',
    'peak_window_end_jd',
    'fit_ok',
    'selection_mode',
    'seed_method',
    'best_model',
    'fit_reduced_chi2',
    'log10_delta_bic_vs_flat',
    'log10_delta_bic_vs_gaussian',
    'log10_delta_bic_vs_fred',
    'log10_delta_bic_vs_best_alt',
    'external_known_microlens',
    'external_best_microlens_source',
    'external_best_microlens_event_id',
    'external_best_microlens_sep_arcsec',
    'external_best_microlens_catalog_t0_time_system',
    'external_best_microlens_catalog_t0_jd',
    'external_best_microlens_delta_t0_days',
    'external_best_microlens_catalog_tE_days',
    'external_best_microlens_catalog_tE_kind',
    'external_best_microlens_delta_tE_days',
    'external_best_microlens_delta_log_tE',
    'gaia_alert_name',
    'gaia_alert_class',
    'gaia_alert_sep_arcsec',
    'gaia_alert_microlens_like',
    'gaia_alert_lookup_status',
    'asassn_ml_match',
    'asassn_ml_name',
    'asassn_ml_sep_arcsec',
    'ogle_ews_event_id',
    'ogle_ews_sep_arcsec',
    'ogle_ews_catalog_t0_time_system',
    'ogle_ews_catalog_t0_jd',
    'ogle_ews_delta_t0_days',
    'ogle_ews_catalog_tE_days',
    'ogle_ews_catalog_tE_kind',
    'ogle_ews_delta_tE_days',
    'ogle_ews_delta_log_tE',
    'kmtnet_event_id',
    'kmtnet_sep_arcsec',
    'kmtnet_catalog_t0_time_system',
    'kmtnet_catalog_t0_jd',
    'kmtnet_delta_t0_days',
    'kmtnet_catalog_tE_days',
    'kmtnet_catalog_tE_kind',
    'kmtnet_delta_tE_days',
    'kmtnet_delta_log_tE',
    'moa_event_id',
    'moa_sep_arcsec',
    'moa_catalog_t0_time_system',
    'moa_catalog_t0_jd',
    'moa_delta_t0_days',
    'moa_catalog_tE_days',
    'moa_catalog_tE_kind',
    'moa_delta_tE_days',
    'moa_delta_log_tE',
    'microlens_match',
    'microlens_catalog',
    'microlens_name',
    'microlens_alt_name',
    'microlens_te_days',
    'microlens_sep_arcsec',
    'vetting_likely_known',
    'catalog_source',
    'nearest_simbad_object',
    'simbad_otype',
    'simbad_sep_arcsec',
    'nearest_vsx_object',
    'vsx_class',
    'vsx_sep_arcsec',
    'fit_warning',
]
raw_delta_bic_cols = {'delta_bic_vs_flat', 'delta_bic_vs_gaussian', 'delta_bic_vs_fred', 'delta_bic_vs_best_alt'}
ordered_master_cols = [
    *master_cols,
    *[col for col in master_df.columns if col not in master_cols and col not in raw_delta_bic_cols],
]

with pd.option_context('display.max_columns', None, 'display.max_colwidth', None):
    display(master_df[ordered_master_cols])

external_match_details_df = external_match_details_df.sort_values(['candidate_id', 'match_rank', 'source_match_rank']).reset_index(drop=True)
with pd.option_context('display.max_columns', None, 'display.max_colwidth', None):
    display(external_match_details_df[EXTERNAL_MATCH_DETAIL_COLS] if not external_match_details_df.empty else pd.DataFrame(columns=EXTERNAL_MATCH_DETAIL_COLS))

gaia_alert_matches_df = master_df.loc[
    master_df['gaia_alert_name'].fillna('').astype(str) != '',
    ['candidate_id', 'gaia_alert_name', 'gaia_alert_class', 'gaia_alert_sep_arcsec', 'gaia_alert_microlens_like', 'gaia_alert_lookup_status'],
].copy()
with pd.option_context('display.max_columns', None, 'display.max_colwidth', None):
    display(gaia_alert_matches_df if not gaia_alert_matches_df.empty else pd.DataFrame(columns=['candidate_id', 'gaia_alert_name', 'gaia_alert_class', 'gaia_alert_sep_arcsec', 'gaia_alert_microlens_like', 'gaia_alert_lookup_status']))


,candidate_id,asassn_source_id,asassn_var_name,asassn_var_type,gaia_dr3_source_id,ra_deg,dec_deg,fit_t0_time_system,fit_t0_jd_minus_2450000,fit_t0_jd,...,microlens_sep_arcsec,vetting_likely_known,catalog_source,nearest_simbad_object,simbad_otype,simbad_sep_arcsec,nearest_vsx_object,vsx_class,vsx_sep_arcsec,fit_warning
0,103079263205,103079263205,,,465382899345974272,36.386244,60.713969,JD-2450000,9095.985516,2.459096e+06,...,None,True,,LS I +60 231,*,0.013,,,NaN,"best_model_not_paczynski,not_preferred_vs_alt"
1,120259784233,120259784233,,,243508157308217856,55.707807,43.213972,JD-2450000,9933.294710,2.459933e+06,...,None,False,,UCAC4 667-019722,SB*,0.026,,,NaN,"best_model_not_paczynski,not_preferred_vs_alt,..."
2,171799355659,171799355659,,,1820343483813175168,299.375675,16.872906,JD-2450000,10173.940166,2.460174e+06,...,None,False,,,,NaN,,,NaN,
3,188979054063,188979054063,,,3424491877389527168,90.240983,22.838601,JD-2450000,8646.019836,2.458646e+06,...,None,True,,NSV 2776,V*,0.035,,,0.269622,"best_model_not_paczynski,not_preferred_vs_alt,..."
4,25771219762,25771219762,,,2068665195625680128,304.451198,41.833794,JD-2450000,10468.289002,2.460468e+06,...,None,False,,TYC 3159-1869-1,*,0.095,,,NaN,"best_model_not_paczynski,not_preferred_vs_alt"
5,326418117943,326418117943,,,4152507736227319040,276.939138,-13.206499,JD-2450000,9521.683459,2.459522e+06,...,None,False,,,,NaN,,,NaN,"best_model_not_paczynski,not_preferred_vs_alt,..."
6,34360800532,34360800532,,,2165231827256902144,316.545347,47.767499,JD-2450000,8914.089721,2.458914e+06,...,None,False,,Gaia DR3 2165231827256902144,*,0.086,,,NaN,"best_model_not_paczynski,not_preferred_vs_alt,..."
7,472447489028,472447489028,,,4057362975065130368,268.258316,-28.768801,JD-2450000,9391.102841,2.459391e+06,...,None,False,,,,NaN,,,NaN,"best_model_not_paczynski,not_preferred_vs_alt"
8,481036788325,481036788325,,,5972620146499838592,259.539029,-39.297612,JD-2450000,8320.336227,2.458320e+06,...,None,True,,Gaia DR3 5972620142171494016,EB*,4.573,,,NaN,t0_near_bound
9,489626721133,489626721133,,,2935004801250074496,107.945416,-16.802683,JD-2450000,9168.402161,2.459168e+06,...,None,False,,,,NaN,,,NaN,


In [6]:
for result in fit_results:
    print(_format_known_microlens_status(result))
    print(_format_parallax_status(result))
    summary = result['summary']
    if summary.get('external_best_microlens_event_id'):
        sep = summary.get('external_best_microlens_sep_arcsec')
        delta_t0 = summary.get('external_best_microlens_delta_t0_days')
        delta_tE = summary.get('external_best_microlens_delta_tE_days')
        sep_text = f'{sep:.3f}"' if sep is not None and np.isfinite(sep) else 'nan'
        delta_t0_text = f'{delta_t0:.2f} d' if delta_t0 is not None and np.isfinite(delta_t0) else 'n/a'
        delta_tE_text = f'{delta_tE:.2f} d' if delta_tE is not None and np.isfinite(delta_tE) else 'n/a'
        print(
            '  external_catalog_best: '
            f"{summary.get('external_best_microlens_source', '')} {summary.get('external_best_microlens_event_id', '')} | "
            f'sep={sep_text} | dt0={delta_t0_text} | dtE={delta_tE_text}'
        )
    elif summary.get('gaia_alert_name'):
        sep = summary.get('gaia_alert_sep_arcsec')
        sep_text = f'{sep:.3f}"' if sep is not None and np.isfinite(sep) else 'nan'
        print(
            '  gaia_alert_match: '
            f"{summary.get('gaia_alert_name', '')} [{summary.get('gaia_alert_class', '') or 'unknown'}] | sep={sep_text}"
        )
    else:
        print('  external_catalog_best: none within the configured cone-match radius')
    fig, _axes = plot_candidate_fit(result)
    tE = result['summary']['reported_tE_days']
    if not __import__('numpy').isfinite(tE):
        tE = result['summary']['raw_paczynski_tE_days']
    cand_id = result['summary']['candidate_id']
    filename = f"{cand_id}_{tE:.3f}.pdf"
    fig.savefig(filename, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved {filename}")
    plt.close(fig)


KeyError: 'simbad_main_id'

In [7]:
OUT_CSV = DB_PATH.parent / 'march18_einstein_crossing_times.csv'
master_df.to_csv(OUT_CSV, index=False)
OUT_CSV


PosixPath('/home/calder/code/malca/output/runs/runs_march18_bundle_all/review/march18_einstein_crossing_times.csv')

In [ ]:
from IPython.display import HTML

html_links = []
for _, row in master_df.iterrows():
    ra, dec, cid = row['ra_deg'], row['dec_deg'], row['candidate_id']
    if pd.notna(ra) and pd.notna(dec):
        url = f"http://vizier.cds.unistra.fr/viz-bin/VizieR-5?-source=ALL&-c={ra:.5f}{dec:+.5f}&-c.rs=10&-out.add=_r&-sort=_r"
        html_links.append(f'<a href="{url}" target="_blank">{cid} (VizieR)</a>')

display(HTML("<br>".join(html_links)))
